# Notebook 09: Final Publication Outputs

## Purpose

Assemble the final manuscript tables and figures directly from the completed analytical outputs, apply consistent publication formatting, package main and appendix artifacts, and write reproducibility manifests and metadata.

## Inputs

- Completed primary outputs from Notebooks 01--07
- Aggregate sensitivity outputs from Notebook 08

## Outputs

- Main-paper Tables 1--3 and Figures 1--3
- Appendix Tables A1--A3 and Figures A1--A4
- Paper-ready and final-output manifests
- `data/processed/final_outputs_metadata.json`

## Dependencies

Run Notebooks 01--08 first. This is the final notebook in the documented pipeline.

> **Repository policy:** Notebook outputs and execution counts are cleared in the public source files. Run the notebooks in the documented order to regenerate all results.

## 1. Setup

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import shutil
import sys

import joblib
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


try:
    import interpret
except ImportError as exc:
    raise ImportError(
        "Notebook 09 requires the same InterpretML installation as Notebook 05. "
        "Install `interpret-core==0.7.8`, restart the kernel, and run again."
    ) from exc

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 300)
pd.set_option("display.width", 220)

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("InterpretML:", interpret.__version__)
print("Matplotlib:", matplotlib.__version__)


## 2. Configuration

The required no-race analysis is enabled by default.

For a quick technical test, temporarily set the environment variable `NOTEBOOK08_BOOTSTRAP_REPLICATES` below 1,000. Final saved results should use 1,000 replicates.

In [ ]:
RANDOM_STATE = 26
N_SPLITS = 5
INNER_THRESHOLD_SPLITS = 3
TARGET_SENSITIVITY_LEVELS = [0.70, 0.80, 0.90]
PRIMARY_TARGET_SENSITIVITY = 0.80
N_BOOTSTRAP = int(os.environ.get(
    "NOTEBOOK09_BOOTSTRAP_REPLICATES",
    os.environ.get("NOTEBOOK08_BOOTSTRAP_REPLICATES", "1000"),
))

RUN_RACE_OMISSION_ANALYSIS = True

TARGET_COLUMNS = [
    "self_reported_prior_diagnosis",
    "current_hba1c_ge_6_5",
]
TARGET_DISPLAY_NAMES = {
    "self_reported_prior_diagnosis": "Prior reported clinician diagnosis",
    "current_hba1c_ge_6_5": "Current HbA1c at least 6.5%",
}
MODEL_NAMES = ["logistic", "ebm"]
MODEL_DISPLAY_NAMES = {
    "logistic": "Logistic regression",
    "ebm": "Explainable Boosting Machine",
}

CONTINUOUS_PREDICTORS = ["age", "bmi", "income_poverty_ratio"]
PRIMARY_CATEGORICAL_PREDICTORS = ["sex", "race_ethnicity", "insurance_history"]
NO_RACE_CATEGORICAL_PREDICTORS = ["sex", "insurance_history"]
PRIMARY_PREDICTORS = CONTINUOUS_PREDICTORS + PRIMARY_CATEGORICAL_PREDICTORS
NO_RACE_PREDICTORS = CONTINUOUS_PREDICTORS + NO_RACE_CATEGORICAL_PREDICTORS

SEX_LEVELS = ["Female", "Male"]
RACE_ETHNICITY_LEVELS = [
    "Mexican American",
    "Other Hispanic",
    "Non-Hispanic White",
    "Non-Hispanic Black",
    "Non-Hispanic Asian",
    "Other or multiracial",
]
INSURANCE_HISTORY_LEVELS = [
    "Continuously insured",
    "Currently insured, past-year gap",
    "Currently uninsured",
]
INCOME_GROUP_LEVELS = ["Below 1", "1 to below 2", "2 to below 4", "4 or higher"]

REFERENCE_CATEGORIES = {
    "sex": "Female",
    "race_ethnicity": "Non-Hispanic White",
    "income_poverty_group": "4 or higher",
    "insurance_history": "Continuously insured",
}
SUBGROUP_LEVELS = {
    "sex": SEX_LEVELS,
    "race_ethnicity": RACE_ETHNICITY_LEVELS,
    "income_poverty_group": INCOME_GROUP_LEVELS,
    "insurance_history": INSURANCE_HISTORY_LEVELS,
}
THRESHOLD_METRICS = [
    "false_negative_rate",
    "false_positive_rate",
    "positive_prediction_rate",
]

print("Bootstrap replicates:", N_BOOTSTRAP)
print("Required race-omission analysis:", RUN_RACE_OMISSION_ANALYSIS)


## 3. Project paths and upstream files

The notebook uses the same project structure as Notebooks 01–07. It first checks the expected relative path and then searches for the same basename while excluding copied final outputs.

In [ ]:
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
TABLE_DIR = PROJECT_DIR / "outputs" / "tables"
FIGURE_DIR = PROJECT_DIR / "outputs" / "figures"
MODEL_DIR = PROJECT_DIR / "outputs" / "models"
FINAL_DIR = PROJECT_DIR / "outputs" / "final"
FINAL_MAIN_DIR = FINAL_DIR / "main"
FINAL_APPENDIX_DIR = FINAL_DIR / "appendix"

for directory in [
    PROCESSED_DIR, TABLE_DIR, FIGURE_DIR, MODEL_DIR,
    FINAL_MAIN_DIR, FINAL_APPENDIX_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

def resolve_project_file(relative_path: str, required: bool = True):
    expected = PROJECT_DIR / relative_path
    if expected.exists():
        return expected

    basename = Path(relative_path).name
    matches = [
        path for path in PROJECT_DIR.rglob(basename)
        if "outputs/final" not in path.as_posix()
    ]
    preferred = [
        path for path in matches
        if path.as_posix().endswith(relative_path.replace("\\", "/"))
    ]
    if len(preferred) == 1:
        return preferred[0]
    if len(matches) == 1:
        return matches[0]
    if len(matches) > 1:
        raise RuntimeError(
            f"Multiple files named {basename!r} were found:\n"
            + "\n".join(f"- {path}" for path in matches)
        )
    if required:
        raise FileNotFoundError(
            f"Required file not found: {expected}\n"
            "Run the upstream notebook that creates it before Notebook 09."
        )
    return None

def load_json(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as file:
        return json.load(file)

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as file:
        for block in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

LABELLED_DATA_PATH = resolve_project_file(
    "data/processed/nhanes_diabetes_complete_case_labeled.csv"
)
ANALYSIS_BASE_PATH = resolve_project_file(
    "data/processed/nhanes_diabetes_analysis_base.csv"
)
FOLD_ASSIGNMENT_PATH = resolve_project_file(
    "data/processed/primary_cv_fold_assignments.csv"
)
LOGISTIC_OOF_PATH = resolve_project_file(
    "data/processed/logistic_oof_predictions.csv"
)
PRIMARY_LOGISTIC_EFFECT_PATH = resolve_project_file(
    "outputs/tables/logistic_probability_effects_with_bootstrap_intervals.csv"
)
EBM_OOF_PATH = resolve_project_file(
    "data/processed/ebm_oof_predictions.csv"
)
THRESHOLDED_LONG_PATH = resolve_project_file(
    "data/processed/thresholded_oof_predictions_long.csv"
)
SAMPLE_METADATA_PATH = resolve_project_file("data/processed/sample_metadata.json")
DESCRIPTIVE_METADATA_PATH = resolve_project_file(
    "data/processed/descriptive_analysis_metadata.json"
)
LOGISTIC_METADATA_PATH = resolve_project_file(
    "data/processed/logistic_regression_metadata.json"
)
EBM_METADATA_PATH = resolve_project_file(
    "data/processed/ebm_analysis_metadata.json"
)
PERFORMANCE_METADATA_PATH = resolve_project_file(
    "data/processed/performance_and_thresholds_metadata.json"
)
FAIRNESS_METADATA_PATH = resolve_project_file(
    "data/processed/fairness_and_subgroup_metadata.json",
    required=False,
)

print("Project directory:", PROJECT_DIR)
print("Analytic data:", LABELLED_DATA_PATH)
print("Fixed folds:", FOLD_ASSIGNMENT_PATH)


## 4. Load and validate the completed primary analysis

Notebook 08 does not recreate the analytic sample or the primary folds. All participant-level inputs are aligned by ID and checked before sensitivity models are fitted.

In [ ]:
data = pd.read_csv(LABELLED_DATA_PATH)
analysis_base = pd.read_csv(ANALYSIS_BASE_PATH)
fold_assignments = pd.read_csv(FOLD_ASSIGNMENT_PATH)
logistic_oof_raw = pd.read_csv(LOGISTIC_OOF_PATH)
primary_logistic_effects = pd.read_csv(
    PRIMARY_LOGISTIC_EFFECT_PATH
)
ebm_oof_raw = pd.read_csv(EBM_OOF_PATH)
primary_thresholded_long = pd.read_csv(THRESHOLDED_LONG_PATH)

sample_metadata = load_json(SAMPLE_METADATA_PATH)
descriptive_metadata = load_json(DESCRIPTIVE_METADATA_PATH)
logistic_metadata = load_json(LOGISTIC_METADATA_PATH)
ebm_metadata = load_json(EBM_METADATA_PATH)
performance_metadata = load_json(PERFORMANCE_METADATA_PATH)
fairness_metadata = (
    load_json(FAIRNESS_METADATA_PATH)
    if FAIRNESS_METADATA_PATH is not None
    else {}
)

required_columns = {
    "id", "joint_label_code", "age", "bmi", "income_poverty_ratio",
    "sex", "race_ethnicity", "insurance_history", "mec_exam_weight",
    *TARGET_COLUMNS,
}
missing = required_columns.difference(data.columns)
if missing:
    raise ValueError(f"Analytic data is missing columns: {sorted(missing)}")
if data["id"].duplicated().any() or fold_assignments["id"].duplicated().any():
    raise ValueError("Participant IDs must be unique.")
if set(data["id"]) != set(fold_assignments["id"]):
    raise ValueError("Analytic IDs and saved fold-assignment IDs differ.")

if "cv_fold" in data.columns:
    check = data[["id", "cv_fold"]].merge(
        fold_assignments[["id", "cv_fold"]],
        on="id",
        suffixes=("_data", "_file"),
        validate="one_to_one",
    )
    if not (
        check["cv_fold_data"].astype(int)
        == check["cv_fold_file"].astype(int)
    ).all():
        raise ValueError("Embedded folds differ from the saved primary folds.")
else:
    data = data.merge(
        fold_assignments[["id", "cv_fold"]],
        on="id",
        how="left",
        validate="one_to_one",
    )

data["cv_fold"] = data["cv_fold"].astype(int)
if sorted(data["cv_fold"].unique()) != list(range(1, N_SPLITS + 1)):
    raise ValueError("The expected five fixed folds are not present.")

data["income_poverty_group"] = pd.cut(
    data["income_poverty_ratio"],
    bins=[-np.inf, 1, 2, 4, np.inf],
    labels=INCOME_GROUP_LEVELS,
    right=False,
).astype(str)

for column, levels in {
    "sex": SEX_LEVELS,
    "race_ethnicity": RACE_ETHNICITY_LEVELS,
    "insurance_history": INSURANCE_HISTORY_LEVELS,
    "income_poverty_group": INCOME_GROUP_LEVELS,
}.items():
    unexpected = set(data[column].dropna().astype(str)).difference(levels)
    if unexpected:
        raise ValueError(f"Unexpected levels in {column}: {sorted(unexpected)}")

for target in TARGET_COLUMNS:
    if set(data[target].astype(int).unique()) != {0, 1}:
        raise ValueError(f"{target} is not a complete binary target.")

if data[PRIMARY_PREDICTORS + TARGET_COLUMNS].isna().any().any():
    raise ValueError("Primary predictors or targets contain missing values.")
if len(data) != int(descriptive_metadata.get("complete_case_n", len(data))):
    raise ValueError("Sample size differs from Notebook 03 metadata.")

print("Adult analysis base:", len(analysis_base))
print("Primary analytic sample:", len(data))
print("Fixed folds:", sorted(data["cv_fold"].unique()))
print(data["joint_label_code"].value_counts().sort_index())


## 5. Shared metric, weighting, threshold, and alignment helpers

In [ ]:
def safe_divide(numerator, denominator):
    numerator = np.asarray(numerator, dtype=float)
    denominator = np.asarray(denominator, dtype=float)
    result = np.full(np.broadcast(numerator, denominator).shape, np.nan)
    np.divide(numerator, denominator, out=result, where=denominator != 0)
    return float(result) if result.ndim == 0 else result

def percentile_interval(values):
    finite = np.asarray(values, dtype=float)
    finite = finite[np.isfinite(finite)]
    if finite.size == 0:
        return np.nan, np.nan
    return tuple(np.percentile(finite, [2.5, 97.5]).astype(float))

def threshold_metrics(outcome, predicted_class):
    y = np.asarray(outcome, dtype=int)
    pred = np.asarray(predicted_class, dtype=int)
    tp = int(((y == 1) & (pred == 1)).sum())
    fn = int(((y == 1) & (pred == 0)).sum())
    fp = int(((y == 0) & (pred == 1)).sum())
    tn = int(((y == 0) & (pred == 0)).sum())
    return {
        "positive_n": int((y == 1).sum()),
        "negative_n": int((y == 0).sum()),
        "predicted_positive_n": int((pred == 1).sum()),
        "false_negative_rate": safe_divide(fn, tp + fn),
        "false_positive_rate": safe_divide(fp, fp + tn),
        "sensitivity": safe_divide(tp, tp + fn),
        "specificity": safe_divide(tn, tn + fp),
        "positive_prediction_rate": float(pred.mean()),
    }

def weighted_mean(values, weights):
    values = pd.to_numeric(values, errors="coerce").to_numpy(float)
    weights = pd.to_numeric(weights, errors="coerce").to_numpy(float)
    valid = np.isfinite(values) & np.isfinite(weights) & (weights > 0)
    return float(np.average(values[valid], weights=weights[valid])) if valid.any() else np.nan

def weighted_sd(values, weights):
    values = pd.to_numeric(values, errors="coerce").to_numpy(float)
    weights = pd.to_numeric(weights, errors="coerce").to_numpy(float)
    valid = np.isfinite(values) & np.isfinite(weights) & (weights > 0)
    if not valid.any():
        return np.nan
    mean = np.average(values[valid], weights=weights[valid])
    return float(np.sqrt(np.average((values[valid] - mean) ** 2, weights=weights[valid])))

def weighted_quantile(values, weights, quantile):
    values = pd.to_numeric(values, errors="coerce").to_numpy(float)
    weights = pd.to_numeric(weights, errors="coerce").to_numpy(float)
    valid = np.isfinite(values) & np.isfinite(weights) & (weights > 0)
    if not valid.any():
        return np.nan
    values, weights = values[valid], weights[valid]
    order = np.argsort(values)
    values, weights = values[order], weights[order]
    cumulative = np.cumsum(weights)
    position = min(np.searchsorted(cumulative, quantile * cumulative[-1]), len(values) - 1)
    return float(values[position])

def infer_probability_column(frame, target, metadata):
    mapping = metadata.get("oof_probability_columns", {})
    if isinstance(mapping, dict):
        candidate = mapping.get(target)
        if isinstance(candidate, str) and candidate in frame.columns:
            return candidate
    candidates = [
        f"oof_probability_{target}",
        f"{target}_oof_probability",
        f"probability_{target}",
        f"{target}_probability",
    ]
    for candidate in candidates:
        if candidate in frame.columns:
            return candidate
    semantic = [
        column for column in frame.columns
        if target in column
        and ("probability" in column.lower() or "prediction" in column.lower())
        and "class" not in column.lower()
    ]
    if len(semantic) == 1:
        return semantic[0]
    raise KeyError(f"Could not identify OOF probability column for {target}.")

def align_oof(frame, metadata, model_name):
    if frame["id"].duplicated().any():
        raise ValueError(f"{model_name} OOF predictions contain duplicate IDs.")
    aligned = data[["id", "cv_fold", *TARGET_COLUMNS]].copy()
    for target in TARGET_COLUMNS:
        column = infer_probability_column(frame, target, metadata)
        aligned = aligned.merge(
            frame[["id", column]].rename(
                columns={column: f"{model_name}__{target}__probability"}
            ),
            on="id",
            how="left",
            validate="one_to_one",
        )
    return aligned

primary_logistic_probabilities = align_oof(logistic_oof_raw, logistic_metadata, "logistic")
primary_ebm_probabilities = align_oof(ebm_oof_raw, ebm_metadata, "ebm")
primary_probabilities = primary_logistic_probabilities.merge(
    primary_ebm_probabilities[
        ["id", *[f"ebm__{target}__probability" for target in TARGET_COLUMNS]]
    ],
    on="id",
    how="left",
    validate="one_to_one",
)
probability_columns = [
    f"{model}__{target}__probability"
    for model in MODEL_NAMES for target in TARGET_COLUMNS
]
if primary_probabilities[probability_columns].isna().any().any():
    raise ValueError("At least one primary OOF probability is missing.")
if not (
    (primary_probabilities[probability_columns] >= 0)
    & (primary_probabilities[probability_columns] <= 1)
).all().all():
    raise ValueError("At least one primary OOF probability lies outside [0, 1].")

print("Aligned primary OOF probabilities:", primary_probabilities.shape)


## Load aggregate sensitivity outputs from Notebook 08

Only saved aggregate tables are loaded here. Participant-level primary predictions continue to come from the established upstream analysis files.

In [ ]:
SELECTION_SUMMARY_PATH = resolve_project_file(
    "outputs/tables/sensitivity_complete_case_selection_summary.csv"
)
EXCLUSION_REASONS_PATH = resolve_project_file(
    "outputs/tables/sensitivity_complete_case_exclusion_reasons.csv"
)
WEIGHTING_SENSITIVITY_PATH = resolve_project_file(
    "outputs/tables/sensitivity_weighted_unweighted_descriptive_comparison.csv"
)
PRIMARY_THRESHOLD_ABSOLUTE_PATH = resolve_project_file(
    "outputs/tables/threshold_sensitivity_absolute_subgroup_metrics.csv"
)
PRIMARY_THRESHOLD_GAPS_PATH = resolve_project_file(
    "outputs/tables/threshold_sensitivity_reference_gaps.csv"
)
THRESHOLD_STABILITY_PATH = resolve_project_file(
    "outputs/tables/threshold_sensitivity_stability_summary.csv"
)
RACE_OMISSION_PERFORMANCE_PATH = resolve_project_file(
    "outputs/tables/race_omission_probability_performance_with_intervals.csv"
)
RACE_OMISSION_EFFECT_PATH = resolve_project_file(
    "outputs/tables/race_omission_probability_effect_comparison.csv"
)
RACE_OMISSION_FAIRNESS_PATH = resolve_project_file(
    "outputs/tables/race_omission_subgroup_changes_with_intervals.csv"
)

selection_summary = pd.read_csv(SELECTION_SUMMARY_PATH)
exclusion_reasons = pd.read_csv(EXCLUSION_REASONS_PATH)
weighted_unweighted_sensitivity = pd.read_csv(WEIGHTING_SENSITIVITY_PATH)
primary_threshold_absolute = pd.read_csv(PRIMARY_THRESHOLD_ABSOLUTE_PATH)
primary_threshold_gaps = pd.read_csv(PRIMARY_THRESHOLD_GAPS_PATH)
threshold_stability = pd.read_csv(THRESHOLD_STABILITY_PATH)
race_omission_performance_with_intervals = pd.read_csv(
    RACE_OMISSION_PERFORMANCE_PATH
)
race_omission_effect_comparison = pd.read_csv(
    RACE_OMISSION_EFFECT_PATH
)
race_omission_fairness_with_intervals = pd.read_csv(
    RACE_OMISSION_FAIRNESS_PATH
)

PRIMARY_LOGISTIC_MODEL_PATHS = {
    target: resolve_project_file(f"outputs/models/logistic_{target}.joblib")
    for target in TARGET_COLUMNS
}
PRIMARY_EBM_MODEL_PATHS = {
    target: resolve_project_file(f"outputs/models/ebm_{target}.joblib")
    for target in TARGET_COLUMNS
}
primary_logistic_models = {
    target: joblib.load(path)
    for target, path in PRIMARY_LOGISTIC_MODEL_PATHS.items()
}
primary_ebm_models = {
    target: joblib.load(path)
    for target, path in PRIMARY_EBM_MODEL_PATHS.items()
}

def keyed_threshold_predictions(frame):
    return {
        (model, target, float(sensitivity)): group.sort_values("id")
        for (model, target, sensitivity), group in frame.groupby(
            ["model", "target", "target_training_sensitivity"],
            observed=True,
        )
    }

def metrics_by_code(outcome, predicted, code, level_n):
    y = np.asarray(outcome, dtype=int)
    pred = np.asarray(predicted, dtype=int)
    code = np.asarray(code, dtype=int)
    positive = np.bincount(
        code, weights=(y == 1).astype(float), minlength=level_n
    )
    negative = np.bincount(
        code, weights=(y == 0).astype(float), minlength=level_n
    )
    fn = np.bincount(
        code,
        weights=((y == 1) & (pred == 0)).astype(float),
        minlength=level_n,
    )
    fp = np.bincount(
        code,
        weights=((y == 0) & (pred == 1)).astype(float),
        minlength=level_n,
    )
    predicted_positive = np.bincount(
        code, weights=(pred == 1).astype(float), minlength=level_n
    )
    total = np.bincount(code, minlength=level_n).astype(float)
    return np.column_stack([
        safe_divide(fn, positive),
        safe_divide(fp, negative),
        safe_divide(predicted_positive, total),
    ])

metric_position = {
    "false_negative_rate": 0,
    "false_positive_rate": 1,
    "positive_prediction_rate": 2,
}

print("Loaded Notebook 08 aggregate sensitivity outputs.")
print("Primary threshold rows:", len(primary_thresholded_long))
print("Threshold-stability rows:", len(threshold_stability))


## Publication-ready tables and figures

The paper-ready package is deliberately compact and visually consistent.

## Main paper

- **Table 1:** analytic sample and target prevalence;
- **Table 2:** numerical concordance of the two operational targets;
- **Table 3:** primary out-of-fold predictive performance;
- **Figure 1:** logistic probability-scale contrasts;
- **Figure 2:** EBM shape functions for the continuous predictors;
- **Figure 3:** subgroup false-negative-rate gaps with bootstrap confidence intervals.

## Appendix

- **Figure A1:** analytic-sample construction;
- **Table A1:** subgroup false-negative-rate gaps with confidence intervals;
- **Table A2:** threshold-sensitive subgroup false-negative-rate gaps across the 70%, 80%, and 90% sensitivity operating points;
- **Table A3:** target-positive subgroup counts used as FNR denominators;
- **Figure A2:** heatmap overview of subgroup false-negative-rate gaps;
- **Figure A3:** out-of-fold calibration curves;
- **Figure A4:** sensitivity to omitting race/ethnicity;
- the complete threshold, weighting, missing-data, model-class, and race-omission outputs.

## Appendix Figure A1 — analytic-sample flow

In [ ]:
# Appendix Figure A1 — analytic-sample construction
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

adult_n = int(
    selection_summary.loc[
        selection_summary["stage"] == "Adult analysis base",
        "n",
    ].iloc[0]
)
analytic_n = int(
    selection_summary.loc[
        selection_summary["stage"] == "Primary analytic sample",
        "n",
    ].iloc[0]
)
excluded_n = adult_n - analytic_n

reason_lookup = exclusion_reasons.set_index("reason")

def exclusion_count(reason):
    if reason not in reason_lookup.index:
        return 0
    return int(
        reason_lookup.loc[
            reason,
            "excluded_n_with_reason",
        ]
    )

pregnancy_n = exclusion_count("Confirmed current pregnancy")
predictor_missing_n = exclusion_count(
    "Missing at least one predictor"
)
target_missing_n = exclusion_count(
    "Missing at least one target"
)
survey_invalid_n = exclusion_count(
    "Missing or invalid required examination/survey field"
)

# Use a compact canvas because the explanatory note is kept in the
# manuscript caption rather than repeated inside the figure.
figure, axis = plt.subplots(figsize=(10.2, 3.8))
axis.set_xlim(0, 10)
axis.set_ylim(1.75, 5.10)
axis.axis("off")

def flow_box(
    x,
    y,
    width,
    height,
    heading,
    value,
    facecolor,
    edgecolor,
):
    patch = FancyBboxPatch(
        (x, y),
        width,
        height,
        boxstyle="round,pad=0.025,rounding_size=0.08",
        linewidth=1.2,
        facecolor=facecolor,
        edgecolor=edgecolor,
    )
    axis.add_patch(patch)
    axis.text(
        x + width / 2,
        y + height * 0.64,
        heading,
        ha="center",
        va="center",
        fontsize=11.5,
        fontweight="semibold",
    )
    axis.text(
        x + width / 2,
        y + height * 0.31,
        value,
        ha="center",
        va="center",
        fontsize=10.5,
    )

flow_box(
    0.55,
    3.65,
    3.25,
    1.25,
    "Adult analysis base",
    f"n = {adult_n:,}",
    "#F4F6F8",
    "#5B6573",
)
flow_box(
    6.20,
    3.65,
    3.25,
    1.25,
    "Primary analytic sample",
    f"n = {analytic_n:,}  ({100 * analytic_n / adult_n:.1f}%)",
    "#EDF4FB",
    "#4C72B0",
)

axis.add_patch(
    FancyArrowPatch(
        (3.95, 4.28),
        (6.05, 4.28),
        arrowstyle="-|>",
        mutation_scale=17,
        linewidth=1.2,
        color="#5B6573",
    )
)
axis.text(
    5.0,
    4.62,
    f"Excluded: n = {excluded_n:,}",
    ha="center",
    va="center",
    fontsize=10.5,
)

reason_data = [
    ("Confirmed pregnancy", pregnancy_n),
    ("Missing predictor(s)", predictor_missing_n),
    ("Missing target(s)", target_missing_n),
    ("Invalid/missing survey field", survey_invalid_n),
]

reason_x = [0.75, 3.05, 5.35, 7.65]
for x_position, (label, count) in zip(reason_x, reason_data):
    axis.text(
        x_position + 0.8,
        2.55,
        label,
        ha="center",
        va="center",
        fontsize=9.5,
        fontweight="semibold",
    )
    axis.text(
        x_position + 0.8,
        2.15,
        f"n = {count:,}",
        ha="center",
        va="center",
        fontsize=10.5,
    )

axis.plot(
    [0.65, 9.35],
    [2.92, 2.92],
    color="#D6D6D6",
    linewidth=0.8,
)
figure.tight_layout(pad=0.2)

appendix_flow_png = (
    FIGURE_DIR / "appendix_figureA1_analytic_sample_flow.png"
)
appendix_flow_pdf = (
    FIGURE_DIR / "appendix_figureA1_analytic_sample_flow.pdf"
)
figure.savefig(
    appendix_flow_png,
    dpi=300,
    bbox_inches="tight",
    pad_inches=0.03,
    facecolor="white",
)
figure.savefig(
    appendix_flow_pdf,
    bbox_inches="tight",
    pad_inches=0.03,
    facecolor="white",
)
plt.close(figure)

appendix_flow_data = pd.DataFrame(
    [
        {"component": "Adult analysis base", "n": adult_n},
        {"component": "Primary analytic sample", "n": analytic_n},
        {"component": "Excluded from primary analysis", "n": excluded_n},
        {"component": "Confirmed pregnancy", "n": pregnancy_n},
        {
            "component": "Missing at least one predictor",
            "n": predictor_missing_n,
        },
        {
            "component": "Missing at least one target",
            "n": target_missing_n,
        },
        {
            "component": "Invalid or missing required survey field",
            "n": survey_invalid_n,
        },
    ]
)
appendix_flow_data.to_csv(
    TABLE_DIR / "appendix_figureA1_analytic_sample_flow_data.csv",
    index=False,
)

print("Saved:", appendix_flow_png)
print("Saved:", appendix_flow_pdf)


## Main-paper tables and figures

In [ ]:
import re
from html import escape
from matplotlib.lines import Line2D

PAPER_READY_DIR = FINAL_MAIN_DIR / "paper_ready"
PAPER_READY_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.edgecolor": "#333333",
    "axes.grid": False,
    "font.family": "DejaVu Sans",
    "font.size": 10.5,
    "axes.titlesize": 13,
    "axes.labelsize": 10.5,
    "xtick.labelsize": 9.5,
    "ytick.labelsize": 9.5,
    "legend.fontsize": 9.5,
    "savefig.facecolor": "white",
    "savefig.edgecolor": "white",
})

MAIN_TARGET_COLORS = {
    TARGET_COLUMNS[0]: "#4C72B0",
    TARGET_COLUMNS[1]: "#DD8452",
}
MODEL_COLORS = {
    "logistic": "#2F5D8A",
    "ebm": "#C17C2F",
}
MODEL_MARKERS = {
    "logistic": "o",
    "ebm": "s",
}


def fmt_percent(value, digits=1):
    if pd.isna(value):
        return "—"
    return f"{100 * float(value):.{digits}f}%"


def fmt_ci(estimate, lower, upper, digits=3, scale=1):
    if pd.isna(estimate):
        return "—"
    est = scale * float(estimate)
    lo = scale * float(lower)
    hi = scale * float(upper)
    return f"{est:.{digits}f} [{lo:.{digits}f}, {hi:.{digits}f}]"


def save_publication_table(
    frame,
    stem,
    index=False,
    caption=None,
    note=None,
):
    csv_path = TABLE_DIR / f"{stem}.csv"
    html_path = TABLE_DIR / f"{stem}.html"
    tex_path = TABLE_DIR / f"{stem}.tex"

    frame.to_csv(csv_path, index=index)

    styler = frame.style
    if not index:
        styler = styler.hide(axis="index")
    styler = (
        styler
        .set_table_styles([
            {
                "selector": "table",
                "props": [
                    ("border-collapse", "collapse"),
                    ("width", "100%"),
                    ("font-family", "Arial, Helvetica, sans-serif"),
                    ("font-size", "12px"),
                    ("color", "#222222"),
                ],
            },
            {
                "selector": "thead th",
                "props": [
                    ("border-top", "2px solid #222222"),
                    ("border-bottom", "1px solid #222222"),
                    ("padding", "8px 10px"),
                    ("text-align", "center"),
                    ("font-weight", "600"),
                    ("background-color", "#FFFFFF"),
                ],
            },
            {
                "selector": "tbody td, tbody th",
                "props": [
                    ("border-bottom", "1px solid #D9D9D9"),
                    ("padding", "7px 10px"),
                    ("vertical-align", "top"),
                ],
            },
            {
                "selector": "tbody tr:last-child td, tbody tr:last-child th",
                "props": [("border-bottom", "2px solid #222222")],
            },
            {
                "selector": "tbody td:not(:first-child)",
                "props": [("text-align", "center")],
            },
            {
                "selector": "tbody th",
                "props": [("text-align", "left"), ("font-weight", "500")],
            },
        ])
    )
    table_html = styler.to_html()

    caption_html = (
        f'<div class="caption">{escape(caption)}</div>'
        if caption else ""
    )
    note_html = (
        f'<div class="note"><strong>Note.</strong> {escape(note)}</div>'
        if note else ""
    )
    html_document = (
        '<!DOCTYPE html>\n'
        '<html lang="en">\n'
        '<head>\n'
        '<meta charset="utf-8">\n'
        f'<title>{escape(caption or stem)}</title>\n'
        '<style>\n'
        'body { max-width: 1200px; margin: 30px auto; padding: 0 24px; background: white; color: #222; }\n'
        '.caption { font: 600 15px Arial, Helvetica, sans-serif; margin: 0 0 12px 0; }\n'
        '.note { font: 11px/1.45 Arial, Helvetica, sans-serif; margin-top: 10px; color: #444; }\n'
        '</style>\n'
        '</head>\n'
        '<body>\n'
        f'{caption_html}\n{table_html}\n{note_html}\n'
        '</body>\n'
        '</html>'
    )
    html_path.write_text(html_document, encoding="utf-8")

    latex = frame.to_latex(
        index=index,
        escape=False,
        multicolumn=True,
        multirow=True,
    )
    # Make all publication-table LaTeX exports compile directly:
    # escape percentages and typeset inequalities in math mode.
    latex = re.sub(r"(?<!\\)%", r"\\%", latex)
    latex = latex.replace(">=", r"$\geq$")
    latex = latex.replace("≥", r"$\geq$")
    latex = re.sub(r"(?<!\\)<", r"$<$", latex)
    tex_path.write_text(latex, encoding="utf-8")

    for path in [csv_path, html_path, tex_path]:
        (PAPER_READY_DIR / path.name).write_bytes(path.read_bytes())

    return {
        "artifact_type": "table",
        "stem": stem,
        "csv": str(csv_path),
        "html": str(html_path),
        "tex": str(tex_path),
    }


def save_publication_figure(figure, stem):
    png_path = FIGURE_DIR / f"{stem}.png"
    pdf_path = FIGURE_DIR / f"{stem}.pdf"
    figure.savefig(png_path, dpi=300, bbox_inches="tight", facecolor="white")
    figure.savefig(pdf_path, bbox_inches="tight", facecolor="white")
    for path in [png_path, pdf_path]:
        (PAPER_READY_DIR / path.name).write_bytes(path.read_bytes())
    plt.close(figure)
    return {
        "artifact_type": "figure",
        "stem": stem,
        "png": str(png_path),
        "pdf": str(pdf_path),
    }


def save_publication_manifest(rows):
    manifest = pd.DataFrame(rows)
    manifest_path = PAPER_READY_DIR / "paper_ready_outputs_manifest.csv"
    manifest.to_csv(manifest_path, index=False)
    return manifest


paper_ready_manifest_rows = []


TARGET_SHORT_NAMES = {
    TARGET_COLUMNS[0]: "Prior diagnosis",
    TARGET_COLUMNS[1]: "HbA1c ≥ 6.5%",
}
MODEL_SHORT_NAMES = {
    "logistic": "Logistic",
    "ebm": "EBM",
}


def clean_axis(axis, grid_axis=None):
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)
    axis.tick_params(length=3, color="#555555")
    if grid_axis is not None:
        axis.grid(
            axis=grid_axis,
            linestyle=":",
            linewidth=0.65,
            color="#D6D6D6",
            zorder=0,
        )


In [ ]:
# Table 1 — Analytic sample and target prevalence
weights = data["mec_exam_weight"]
weighted_q1 = weighted_quantile(data["income_poverty_ratio"], weights, 0.25)
weighted_median = weighted_quantile(data["income_poverty_ratio"], weights, 0.50)
weighted_q3 = weighted_quantile(data["income_poverty_ratio"], weights, 0.75)

rows = []


def add_table1_row(section, characteristic, unweighted, weighted):
    rows.append({
        "Section": section,
        "Characteristic": characteristic,
        "Unweighted analytic sample": unweighted,
        "MEC-weighted complete-case estimate": weighted,
    })


add_table1_row("Sample", "Analytic sample, n", f"{len(data):,}", "—")
add_table1_row(
    "Continuous characteristics",
    "Age, years, mean (SD)",
    f"{data['age'].mean():.1f} ({data['age'].std(ddof=1):.1f})",
    f"{weighted_mean(data['age'], weights):.1f} ({weighted_sd(data['age'], weights):.1f})",
)
add_table1_row(
    "Continuous characteristics",
    "BMI, kg/m², mean (SD)",
    f"{data['bmi'].mean():.1f} ({data['bmi'].std(ddof=1):.1f})",
    f"{weighted_mean(data['bmi'], weights):.1f} ({weighted_sd(data['bmi'], weights):.1f})",
)
add_table1_row(
    "Continuous characteristics",
    "Income-to-poverty ratio, median [IQR]",
    f"{data['income_poverty_ratio'].median():.2f} [{data['income_poverty_ratio'].quantile(0.25):.2f}, {data['income_poverty_ratio'].quantile(0.75):.2f}]",
    f"{weighted_median:.2f} [{weighted_q1:.2f}, {weighted_q3:.2f}]",
)

for level in SEX_LEVELS:
    indicator = (data['sex'] == level).astype(int)
    add_table1_row(
        "Sex",
        level,
        f"{int(indicator.sum()):,} ({100 * indicator.mean():.1f}%)",
        fmt_percent(weighted_mean(indicator, weights), digits=1),
    )

for level in RACE_ETHNICITY_LEVELS:
    indicator = (data['race_ethnicity'] == level).astype(int)
    add_table1_row(
        "Race/ethnicity",
        level,
        f"{int(indicator.sum()):,} ({100 * indicator.mean():.1f}%)",
        fmt_percent(weighted_mean(indicator, weights), digits=1),
    )

for level in INSURANCE_HISTORY_LEVELS:
    indicator = (data['insurance_history'] == level).astype(int)
    add_table1_row(
        "Insurance history",
        level,
        f"{int(indicator.sum()):,} ({100 * indicator.mean():.1f}%)",
        fmt_percent(weighted_mean(indicator, weights), digits=1),
    )

for target in TARGET_COLUMNS:
    indicator = pd.to_numeric(data[target], errors='coerce').fillna(0).astype(int)
    add_table1_row(
        "Operational targets",
        TARGET_DISPLAY_NAMES[target],
        f"{int(indicator.sum()):,} ({100 * indicator.mean():.1f}%)",
        fmt_percent(weighted_mean(indicator, weights), digits=1),
    )

table1_publication = pd.DataFrame(rows).set_index(["Section", "Characteristic"])

paper_ready_manifest_rows.append(
    save_publication_table(
        table1_publication,
        'paper_table1_analytic_sample',
        index=True,
        caption='Table 1. Characteristics of the primary analytic sample',
        note=(
            'Values are mean (SD), median [IQR], or n (%), as indicated. '
            'MEC-weighted estimates are descriptive complete-case estimates. '
            'They use the NHANES MEC examination weight; the displayed uncertainty is not survey-design adjusted.'
        ),
    )
)
table1_publication


In [ ]:
# Table 2 — numerical concordance of the two operational targets
count_matrix = pd.crosstab(
    data['self_reported_prior_diagnosis'],
    data['current_hba1c_ge_6_5'],
).reindex(index=[0, 1], columns=[0, 1], fill_value=0)

row_names = {0: 'No prior reported diagnosis', 1: 'Prior reported diagnosis'}
col_names = {0: 'HbA1c < 6.5%', 1: 'HbA1c ≥ 6.5%'}
total_n = int(count_matrix.to_numpy().sum())

rows = []
for row_value in [0, 1]:
    row = {'Prior diagnosis label': row_names[row_value]}
    for col_value in [0, 1]:
        count = int(count_matrix.loc[row_value, col_value])
        row[col_names[col_value]] = f"{count:,} ({100 * count / total_n:.1f}%)"
    row['Row total'] = (
        f"{int(count_matrix.loc[row_value].sum()):,} "
        f"({100 * count_matrix.loc[row_value].sum() / total_n:.1f}%)"
    )
    rows.append(row)

column_total = {'Prior diagnosis label': 'Column total'}
for col_value in [0, 1]:
    col_count = int(count_matrix[col_value].sum())
    column_total[col_names[col_value]] = f"{col_count:,} ({100 * col_count / total_n:.1f}%)"
column_total['Row total'] = f"{total_n:,} (100.0%)"
rows.append(column_total)

table2_publication = pd.DataFrame(rows)
paper_ready_manifest_rows.append(
    save_publication_table(
        table2_publication,
        'paper_table2_target_concordance',
        index=False,
        caption='Table 2. Concordance of the two operational diabetes targets',
        note='Percentages use the complete primary analytic sample as the denominator.',
    )
)
table2_publication


In [ ]:
# Table 3 — Primary predictive performance
performance_subset = race_omission_performance_with_intervals.copy()
primary_threshold_subset = primary_thresholded_long.loc[
    np.isclose(
        primary_thresholded_long['target_training_sensitivity'],
        PRIMARY_TARGET_SENSITIVITY,
    )
].copy()

threshold_metrics_rows = []
for (model_name, target), group in primary_threshold_subset.groupby(
    ['model', 'target'],
    observed=True,
):
    metrics = threshold_metrics(
        group['observed_outcome'].to_numpy(int),
        group['predicted_class'].to_numpy(int),
    )
    threshold_metrics_rows.append({
        'model': model_name,
        'target': target,
        'specificity_at_primary': metrics['specificity'],
    })
threshold_summary = pd.DataFrame(threshold_metrics_rows)

summary_rows = []
for target in TARGET_COLUMNS:
    prevalence = pd.to_numeric(data[target], errors='coerce').fillna(0).mean()
    for model_name in MODEL_NAMES:
        row = {
            'Target': TARGET_DISPLAY_NAMES[target],
            'Model': MODEL_DISPLAY_NAMES[model_name],
            'Outcome prevalence': fmt_percent(prevalence, digits=1),
        }
        subset = performance_subset.loc[
            (performance_subset['model'] == model_name)
            & (performance_subset['target'] == target)
        ]
        metric_label_map = {
            'roc_auc': 'ROC-AUC',
            'pr_auc': 'Average precision',
            'brier_score': 'Brier score',
        }
        for metric_name, label in metric_label_map.items():
            point = subset.loc[subset['metric'] == metric_name].iloc[0]
            row[label] = fmt_ci(
                point['with_race_estimate'],
                point['with_race_ci_lower'],
                point['with_race_ci_upper'],
                digits=3,
                scale=1,
            )
        threshold_point = threshold_summary.loc[
            (threshold_summary['model'] == model_name)
            & (threshold_summary['target'] == target)
        ].iloc[0]
        row['Specificity at  80% sensitivity'] = fmt_percent(
            threshold_point['specificity_at_primary'],
            digits=1,
        )
        summary_rows.append(row)

table3_publication = pd.DataFrame(summary_rows)
paper_ready_manifest_rows.append(
    save_publication_table(
        table3_publication,
        'paper_table3_primary_performance',
        index=False,
        caption='Table 3. Out-of-fold predictive performance by target and model class',
        note=(
            "Values in brackets are bootstrap 95% confidence intervals for "
            "ROC-AUC, average precision, and Brier score. Average precision "
            "depends on target prevalence and should be interpreted alongside "
            "the reported prevalence. Specificity is reported as a descriptive "
            "point estimate at model- and target-specific thresholds selected "
            "within training data to target 80% sensitivity and evaluated "
            "out of fold."
        ),
    )
)
table3_publication


In [ ]:
# Appendix Table A1 — subgroup FNR gaps with 95% bootstrap confidence intervals
primary_fnr_gap_point = primary_threshold_gaps.loc[
    (primary_threshold_gaps['metric'] == 'false_negative_rate')
    & np.isclose(primary_threshold_gaps['target_training_sensitivity'], PRIMARY_TARGET_SENSITIVITY)
    & (primary_threshold_gaps['subgroup'] != primary_threshold_gaps['reference_group'])
].copy()

primary_threshold_80 = primary_thresholded_long.loc[
    np.isclose(primary_thresholded_long['target_training_sensitivity'], PRIMARY_TARGET_SENSITIVITY)
].copy()
primary_threshold_80_keyed = keyed_threshold_predictions(primary_threshold_80)

ordered_data = data.sort_values('id').reset_index(drop=True)
ordered_ids = ordered_data['id'].to_numpy()
for key, group in primary_threshold_80_keyed.items():
    if not np.array_equal(group['id'].to_numpy(), ordered_ids):
        raise ValueError(f"Primary thresholded IDs are misaligned for {key}.")

rng_primary_gaps = np.random.default_rng(RANDOM_STATE + 8300)
fnr_gap_interval_rows = []
for model_name in MODEL_NAMES:
    for target in TARGET_COLUMNS:
        group = primary_threshold_80_keyed[(model_name, target, float(PRIMARY_TARGET_SENSITIVITY))]
        outcome = group['observed_outcome'].to_numpy(int)
        pred = group['predicted_class'].to_numpy(int)
        for subgroup_variable, levels in SUBGROUP_LEVELS.items():
            codes = pd.Categorical(
                ordered_data[subgroup_variable].astype(str),
                categories=levels,
                ordered=True,
            ).codes
            if (codes < 0).any():
                raise ValueError(f"Unknown subgroup level in {subgroup_variable}.")
            reference = REFERENCE_CATEGORIES[subgroup_variable]
            reference_position = levels.index(reference)
            bootstrap = np.full((N_BOOTSTRAP, len(levels)), np.nan)
            for bootstrap_index in range(N_BOOTSTRAP):
                sampled = rng_primary_gaps.integers(0, len(ordered_data), size=len(ordered_data))
                sampled_metrics = metrics_by_code(
                    outcome[sampled], pred[sampled], codes[sampled], len(levels)
                )
                fnr = sampled_metrics[:, metric_position['false_negative_rate']]
                bootstrap[bootstrap_index, :] = fnr - fnr[reference_position]
            point_subset = primary_fnr_gap_point.loc[
                (primary_fnr_gap_point['model'] == model_name)
                & (primary_fnr_gap_point['target'] == target)
                & (primary_fnr_gap_point['subgroup_variable'] == subgroup_variable)
            ].set_index('subgroup')
            count_subset = primary_threshold_absolute.loc[
                (primary_threshold_absolute['model'] == model_name)
                & (primary_threshold_absolute['target'] == target)
                & np.isclose(primary_threshold_absolute['target_training_sensitivity'], PRIMARY_TARGET_SENSITIVITY)
                & (primary_threshold_absolute['subgroup_variable'] == subgroup_variable)
                & (primary_threshold_absolute['metric'] == 'false_negative_rate')
            ].set_index('subgroup')
            for level_position, level in enumerate(levels):
                if level == reference:
                    continue
                ci_lower, ci_upper = percentile_interval(bootstrap[:, level_position])
                fnr_gap_interval_rows.append({
                    'model': model_name,
                    'target': target,
                    'subgroup_variable': subgroup_variable,
                    'subgroup': level,
                    'reference_group': reference,
                    'gap_estimate': float(point_subset.loc[level, 'gap_from_reference']),
                    'gap_ci_lower': ci_lower,
                    'gap_ci_upper': ci_upper,
                    'subgroup_n': int(count_subset.loc[level, 'subgroup_n']),
                    'positive_n': int(count_subset.loc[level, 'positive_n']),
                })

primary_fnr_gap_intervals = pd.DataFrame(fnr_gap_interval_rows)

label_rows = []
for subgroup_variable, levels in [
    ('sex', SEX_LEVELS),
    ('race_ethnicity', RACE_ETHNICITY_LEVELS),
    ('income_poverty_group', INCOME_GROUP_LEVELS),
    ('insurance_history', INSURANCE_HISTORY_LEVELS),
]:
    reference = REFERENCE_CATEGORIES[subgroup_variable]
    for level in levels:
        if level == reference:
            continue
        label_rows.append((subgroup_variable, level))

pretty_variable_names = {
    'sex': 'Sex',
    'race_ethnicity': 'Race/ethnicity',
    'income_poverty_group': 'Income group',
    'insurance_history': 'Insurance history',
}

appendix_rows = []
for subgroup_variable, level in label_rows:
    row = {
        'Subgroup comparison': (
            f"{pretty_variable_names[subgroup_variable]}: {level} "
            f"(reference: {REFERENCE_CATEGORIES[subgroup_variable]})"
        )
    }
    for model_name in MODEL_NAMES:
        for target in TARGET_COLUMNS:
            subset = primary_fnr_gap_intervals.loc[
                (primary_fnr_gap_intervals['model'] == model_name)
                & (primary_fnr_gap_intervals['target'] == target)
                & (primary_fnr_gap_intervals['subgroup_variable'] == subgroup_variable)
                & (primary_fnr_gap_intervals['subgroup'] == level)
            ].iloc[0]
            column = f"{MODEL_DISPLAY_NAMES[model_name]} | {TARGET_DISPLAY_NAMES[target]}"
            row[column] = fmt_ci(
                subset['gap_estimate'],
                subset['gap_ci_lower'],
                subset['gap_ci_upper'],
                digits=1,
                scale=100,
            )
    appendix_rows.append(row)

appendix_tableA1 = pd.DataFrame(appendix_rows)
paper_ready_manifest_rows.append(
    save_publication_table(
        appendix_tableA1,
        'appendix_tableA1_primary_fnr_gaps_with_ci',
        index=False,
        caption='Appendix Table A1. Subgroup false-negative-rate gaps at the 80% sensitivity operating point',
        note=(
            'Values are percentage-point gaps from the stated reference group with percentile bootstrap 95% confidence intervals. '
            'Positive values indicate a higher false-negative rate than the reference group; negative values indicate a lower false-negative rate.'
        ),
    )
)
appendix_tableA1.head(12)


## Appendix Table A2 — threshold sensitivity of subgroup FNR gaps

This publication-ready appendix table contains the nonreference subgroup
false-negative-rate gap patterns that met the notebook's descriptive
threshold-dependence rule across operating points targeting 70%, 80%, and
90% sensitivity. A pattern is labelled threshold-dependent when its gap
changes sign or spans more than 10 percentage points. This classification
is descriptive rather than a formal statistical test.

In [ ]:
# Appendix Table A2 — threshold-sensitive subgroup FNR-gap patterns
threshold_levels_for_table = [0.70, 0.80, 0.90]

fnr_threshold_gaps = primary_threshold_gaps.loc[
    (primary_threshold_gaps["metric"] == "false_negative_rate")
    & (
        primary_threshold_gaps["subgroup"]
        != primary_threshold_gaps["reference_group"]
    )
].copy()

fnr_threshold_stability = threshold_stability.loc[
    (threshold_stability["metric"] == "false_negative_rate")
    & (
        threshold_stability["subgroup"]
        != threshold_stability["reference_group"]
    )
].copy()

threshold_table_keys = [
    "model",
    "target",
    "subgroup_variable",
    "subgroup",
    "reference_group",
    "metric",
]

threshold_value_table = None
for sensitivity_level in threshold_levels_for_table:
    level_subset = (
        fnr_threshold_gaps.loc[
            np.isclose(
                fnr_threshold_gaps["target_training_sensitivity"],
                sensitivity_level,
            ),
            threshold_table_keys + ["gap_from_reference"],
        ]
        .rename(
            columns={
                "gap_from_reference": (
                    f"fnr_gap_at_{int(100 * sensitivity_level)}"
                )
            }
        )
    )

    if threshold_value_table is None:
        threshold_value_table = level_subset
    else:
        threshold_value_table = threshold_value_table.merge(
            level_subset,
            on=threshold_table_keys,
            how="outer",
            validate="one_to_one",
        )

if threshold_value_table is None:
    raise RuntimeError(
        "No threshold-specific FNR-gap values were available."
    )

appendix_tableA2_source = fnr_threshold_stability.merge(
    threshold_value_table,
    on=threshold_table_keys,
    how="left",
    validate="one_to_one",
)

appendix_tableA2_source = appendix_tableA2_source.loc[
    appendix_tableA2_source[
        "threshold_stability_label"
    ].str.startswith(
        "Threshold-dependent",
        na=False,
    )
].copy()

if appendix_tableA2_source.empty:
    raise RuntimeError(
        "No FNR-gap pattern met the descriptive "
        "threshold-dependence rule."
    )

required_threshold_value_columns = [
    "fnr_gap_at_70",
    "fnr_gap_at_80",
    "fnr_gap_at_90",
]
if appendix_tableA2_source[
    required_threshold_value_columns
].isna().any().any():
    raise ValueError(
        "Appendix Table A2 contains missing threshold-specific "
        "FNR-gap values."
    )

subgroup_dimension_labels = {
    "sex": "Sex",
    "race_ethnicity": "Race/ethnicity",
    "income_poverty_group": "Income group",
    "insurance_history": "Insurance history",
}
subgroup_dimension_order = {
    "sex": 0,
    "race_ethnicity": 1,
    "income_poverty_group": 2,
    "insurance_history": 3,
}
target_table_labels = {
    TARGET_COLUMNS[0]: "Prior diagnosis",
    TARGET_COLUMNS[1]: "Current HbA1c",
}
model_table_labels = {
    "logistic": "Logistic regression",
    "ebm": "EBM",
}

appendix_tableA2_source["dimension_order"] = (
    appendix_tableA2_source["subgroup_variable"]
    .map(subgroup_dimension_order)
)
appendix_tableA2_source = appendix_tableA2_source.sort_values(
    [
        "dimension_order",
        "subgroup",
        "model",
        "target",
    ],
    kind="stable",
).reset_index(drop=True)

appendix_tableA2_source["Subgroup dimension"] = (
    appendix_tableA2_source["subgroup_variable"]
    .map(subgroup_dimension_labels)
)
appendix_tableA2_source["Subgroup comparison"] = (
    appendix_tableA2_source["subgroup"].astype(str)
    + " (reference: "
    + appendix_tableA2_source["reference_group"].astype(str)
    + ")"
)
appendix_tableA2_source["Model"] = (
    appendix_tableA2_source["model"]
    .map(model_table_labels)
)
appendix_tableA2_source["Target"] = (
    appendix_tableA2_source["target"]
    .map(target_table_labels)
)

for sensitivity_level in [70, 80, 90]:
    appendix_tableA2_source[
        f"FNR gap at {sensitivity_level}% sensitivity (pp)"
    ] = appendix_tableA2_source[
        f"fnr_gap_at_{sensitivity_level}"
    ].map(
        lambda value: f"{100 * float(value):+.1f}"
    )

appendix_tableA2_source[
    "Range across operating points (pp)"
] = appendix_tableA2_source["gap_range"].map(
    lambda value: f"{100 * float(value):.1f}"
)
appendix_tableA2_source["Descriptive rule"] = np.where(
    appendix_tableA2_source["gap_sign_changes"],
    "Gap changes sign",
    "Range above 10 pp",
)

appendix_tableA2 = appendix_tableA2_source[
    [
        "Subgroup dimension",
        "Subgroup comparison",
        "Model",
        "Target",
        "FNR gap at 70% sensitivity (pp)",
        "FNR gap at 80% sensitivity (pp)",
        "FNR gap at 90% sensitivity (pp)",
        "Range across operating points (pp)",
        "Descriptive rule",
    ]
].copy()

appendix_tableA2_meta = save_publication_table(
    appendix_tableA2,
    "appendix_tableA2_threshold_sensitivity_fnr_gaps",
    index=False,
)

# LaTeX escaping and inequality formatting are applied centrally by
# save_publication_table for every publication table.

paper_ready_manifest_rows.append(
    appendix_tableA2_meta
)

print(
    "Saved Appendix Table A2 with",
    len(appendix_tableA2),
    "threshold-dependent FNR-gap patterns.",
)
appendix_tableA2


## Appendix Table A3 — target-positive subgroup counts

This table is calculated directly from the complete-case analytic data. The target-positive counts are the denominators used when estimating subgroup false-negative rates.

In [ ]:
# Appendix Table A3 — target-positive subgroup counts used as FNR denominators
subgroup_count_specs = [
    (
        "Sex",
        "sex",
        SEX_LEVELS,
        {
            "Female": "Female (reference)",
            "Male": "Male",
        },
    ),
    (
        "Race/ethnicity",
        "race_ethnicity",
        RACE_ETHNICITY_LEVELS,
        {
            "Mexican American": "Mexican American",
            "Other Hispanic": "Other Hispanic",
            "Non-Hispanic White": "Non-Hispanic White (reference)",
            "Non-Hispanic Black": "Non-Hispanic Black",
            "Non-Hispanic Asian": "Non-Hispanic Asian",
            "Other or multiracial": "Other or multiracial",
        },
    ),
    (
        "Income group",
        "income_poverty_group",
        INCOME_GROUP_LEVELS,
        {
            "Below 1": "IPR < 1",
            "1 to below 2": "IPR 1 to < 2",
            "2 to below 4": "IPR 2 to < 4",
            "4 or higher": "IPR 4 or higher (reference)",
        },
    ),
    (
        "Insurance history",
        "insurance_history",
        INSURANCE_HISTORY_LEVELS,
        {
            "Continuously insured": "Continuously insured (reference)",
            "Currently insured, past-year gap": (
                "Currently insured, past-year gap"
            ),
            "Currently uninsured": "Currently uninsured",
        },
    ),
]

subgroup_positive_rows = []
for dimension, variable, levels, display_labels in subgroup_count_specs:
    dimension_total = 0
    for level in levels:
        mask = data[variable].astype(str).eq(level)
        total_n = int(mask.sum())
        prior_positive_n = int(
            data.loc[mask, TARGET_COLUMNS[0]].astype(int).sum()
        )
        hba1c_positive_n = int(
            data.loc[mask, TARGET_COLUMNS[1]].astype(int).sum()
        )
        if prior_positive_n > total_n or hba1c_positive_n > total_n:
            raise ValueError(
                f"Positive count exceeds subgroup size for {variable}: {level}"
            )
        dimension_total += total_n
        subgroup_positive_rows.append(
            {
                "Dimension": dimension,
                "Group": display_labels[level],
                r"Total \(n\)": f"{total_n:,}",
                r"Prior diagnosis \(n_{+}\)": f"{prior_positive_n:,}",
                r"HbA1c ≥ 6.5% \(n_{+}\)": f"{hba1c_positive_n:,}",
            }
        )
    if dimension_total != len(data):
        raise ValueError(
            f"Subgroup counts for {variable} sum to {dimension_total}, "
            f"not the analytic sample size {len(data)}."
        )

appendix_tableA3 = pd.DataFrame(subgroup_positive_rows)

appendix_tableA3_meta = save_publication_table(
    appendix_tableA3,
    "appendix_tableA3_subgroup_positive_counts",
    index=False,
    caption=(
        "Appendix Table A3. Target-positive subgroup counts used as "
        "denominators in the false-negative-rate analyses"
    ),
    note=(
        "n+ denotes the number of participants positive under the "
        "stated operational target and therefore the denominator from "
        "which the corresponding false-negative rate was estimated. "
        "Counts are identical across model classes because the same "
        "participants and target definitions were used. IPR denotes "
        "income-to-poverty ratio."
    ),
)
paper_ready_manifest_rows.append(appendix_tableA3_meta)

appendix_tableA3


In [ ]:
# Figure 1 — primary logistic probability-scale contrasts
plot_effect_order = [
    "ame_age_1sd",
    "ame_bmi_1sd",
    "ame_income_1sd",
    "adc_sex_male_vs_female",
    "adc_insurance_gap",
    "adc_currently_uninsured",
]
plot_effect_labels = {
    "ame_age_1sd": "Age (+1 SD)",
    "ame_bmi_1sd": "BMI (+1 SD)",
    "ame_income_1sd": "Income-to-poverty ratio (+1 SD)",
    "adc_sex_male_vs_female": "Male vs female",
    "adc_insurance_gap": "Past-year insurance gap",
    "adc_currently_uninsured": "Currently uninsured",
}

required_effect_columns = {
    "target",
    "effect_id",
    "estimate_percentage_points",
    "ci_lower_percentage_points",
    "ci_upper_percentage_points",
}
missing_effect_columns = required_effect_columns.difference(
    primary_logistic_effects.columns
)
if missing_effect_columns:
    raise ValueError(
        "Primary logistic-effect table is missing columns: "
        f"{sorted(missing_effect_columns)}"
    )

plot_data = primary_logistic_effects.loc[
    primary_logistic_effects["effect_id"].isin(plot_effect_order)
].copy()

expected_effect_rows = len(plot_effect_order) * len(TARGET_COLUMNS)
if len(plot_data) != expected_effect_rows:
    observed_pairs = set(
        zip(plot_data["target"], plot_data["effect_id"])
    )
    expected_pairs = {
        (target, effect_id)
        for target in TARGET_COLUMNS
        for effect_id in plot_effect_order
    }
    missing_pairs = sorted(expected_pairs.difference(observed_pairs))
    raise ValueError(
        "Primary logistic-effect table does not contain every "
        f"target-effect combination. Missing: {missing_pairs}"
    )

plot_data["effect_order"] = plot_data["effect_id"].map(
    {
        effect_id: position
        for position, effect_id in enumerate(plot_effect_order)
    }
)
plot_data = plot_data.sort_values(
    ["effect_order", "target"]
)

figure, axis = plt.subplots(figsize=(9.8, 5.9))
base_positions = np.arange(len(plot_effect_order))[::-1]
offsets = {
    TARGET_COLUMNS[0]: -0.13,
    TARGET_COLUMNS[1]: 0.13,
}

for target in TARGET_COLUMNS:
    subset = (
        plot_data.loc[plot_data["target"] == target]
        .set_index("effect_id")
        .reindex(plot_effect_order)
        .reset_index()
    )

    estimate = subset[
        "estimate_percentage_points"
    ].to_numpy(float)
    lower = subset[
        "ci_lower_percentage_points"
    ].to_numpy(float)
    upper = subset[
        "ci_upper_percentage_points"
    ].to_numpy(float)

    axis.errorbar(
        estimate,
        base_positions + offsets[target],
        xerr=np.vstack(
            [
                estimate - lower,
                upper - estimate,
            ]
        ),
        fmt="o",
        markersize=5.8,
        capsize=3,
        color=MAIN_TARGET_COLORS[target],
        ecolor=MAIN_TARGET_COLORS[target],
        linewidth=1.3,
        label=TARGET_SHORT_NAMES[target],
        zorder=3,
    )

axis.axvline(
    0,
    linestyle="--",
    linewidth=1,
    color="#666666",
)
axis.set_yticks(
    base_positions,
    labels=[
        plot_effect_labels[item]
        for item in plot_effect_order
    ],
)
axis.set_xlabel(
    "Average probability change (percentage points)"
)
clean_axis(axis, grid_axis="x")
axis.legend(
    loc="lower right",
    frameon=False,
)
figure.tight_layout()

figure1_meta = save_publication_figure(
    figure,
    "figure1_logistic_probability_effects",
)
paper_ready_manifest_rows.append(figure1_meta)
print(figure1_meta)


In [ ]:
# Figure 2 — EBM shape functions

def extract_ebm_term_curve(
    model,
    feature_name,
    observed_values,
):
    if hasattr(model, "term_names_"):
        term_names = list(model.term_names_)
    elif hasattr(model, "feature_names"):
        term_names = list(model.feature_names)
    else:
        raise AttributeError(
            "Could not identify EBM term names."
        )

    if feature_name not in term_names:
        raise ValueError(
            f"{feature_name!r} not found in EBM term names."
        )

    term_index = term_names.index(feature_name)
    explanation = model.explain_global(name="global")
    term_data = explanation.data(term_index)

    scores = np.asarray(
        term_data["scores"],
        dtype=float,
    ).reshape(-1)
    raw_names = list(term_data.get("names", []))

    numeric_names = pd.to_numeric(
        pd.Series(raw_names),
        errors="coerce",
    ).to_numpy(float)

    if len(numeric_names) == len(scores) + 1:
        edges = (
            pd.Series(numeric_names)
            .interpolate(limit_direction="both")
            .to_numpy(float)
        )
        x_values = (
            edges[:-1] + edges[1:]
        ) / 2
    elif len(numeric_names) == len(scores):
        x_values = (
            pd.Series(numeric_names)
            .interpolate(limit_direction="both")
            .to_numpy(float)
        )
    else:
        parsed_values = []

        for label in map(str, raw_names):
            values = [
                float(value)
                for value in re.findall(
                    r"-?\d+(?:\.\d+)?",
                    label,
                )
            ]

            if len(values) >= 2:
                parsed_values.append(
                    (values[0] + values[1]) / 2
                )
            elif len(values) == 1:
                parsed_values.append(values[0])
            else:
                parsed_values.append(np.nan)

        parsed_values = np.asarray(
            parsed_values,
            dtype=float,
        )

        if len(parsed_values) == len(scores) + 1:
            edges = (
                pd.Series(parsed_values)
                .interpolate(limit_direction="both")
                .to_numpy(float)
            )
            x_values = (
                edges[:-1] + edges[1:]
            ) / 2
        elif len(parsed_values) == len(scores):
            x_values = (
                pd.Series(parsed_values)
                .interpolate(limit_direction="both")
                .to_numpy(float)
            )
        else:
            observed = pd.to_numeric(
                pd.Series(observed_values),
                errors="coerce",
            ).dropna()
            x_values = np.linspace(
                float(observed.min()),
                float(observed.max()),
                len(scores),
            )

    length = min(
        len(x_values),
        len(scores),
    )
    x_values = np.asarray(
        x_values[:length],
        dtype=float,
    )
    scores = np.asarray(
        scores[:length],
        dtype=float,
    )
    finite = (
        np.isfinite(x_values)
        & np.isfinite(scores)
    )

    return (
        pd.DataFrame(
            {
                "x": x_values[finite],
                "y": scores[finite],
            }
        )
        .groupby("x", as_index=False)["y"]
        .mean()
        .sort_values("x")
    )


feature_specs = [
    ("age", "Age (years)", "A", "Age"),
    ("bmi", "BMI (kg/m²)", "B", "BMI"),
    (
        "income_poverty_ratio",
        "Income-to-poverty ratio",
        "C",
        "Income-to-poverty ratio",
    ),
]

figure, axes = plt.subplots(
    1,
    3,
    figsize=(14.8, 4.9),
    sharey=True,
)

for axis, (
    feature_name,
    x_label,
    panel_label,
    panel_title,
) in zip(
    axes,
    feature_specs,
):
    observed = pd.to_numeric(
        data[feature_name],
        errors="coerce",
    ).dropna()

    x_lower, x_upper = observed.quantile(
        [0.025, 0.975]
    )

    for target in TARGET_COLUMNS:
        curve = extract_ebm_term_curve(
            primary_ebm_models[target],
            feature_name,
            data[feature_name],
        )
        curve = curve.loc[
            (curve["x"] >= x_lower)
            & (curve["x"] <= x_upper)
        ]

        axis.plot(
            curve["x"],
            curve["y"],
            linewidth=2.0,
            drawstyle="steps-mid",
            color=MAIN_TARGET_COLORS[target],
            label=TARGET_SHORT_NAMES[target],
        )

    axis.axhline(
        0,
        linestyle="--",
        linewidth=1,
        color="#666666",
    )
    axis.set_xlim(
        x_lower,
        x_upper,
    )
    axis.set_xlabel(x_label)
    axis.set_title(
        f"{panel_label}  {panel_title}",
        loc="left",
        fontweight="semibold",
        pad=9,
    )
    clean_axis(
        axis,
        grid_axis="y",
    )

    rug_values = observed.quantile(
        np.linspace(0.025, 0.975, 70)
    ).to_numpy(float)
    y_min, y_max = axis.get_ylim()
    rug_y = y_min + 0.025 * (y_max - y_min)
    axis.plot(
        rug_values,
        np.repeat(
            rug_y,
            len(rug_values),
        ),
        "|",
        color="#6F6F6F",
        alpha=0.23,
        markersize=5,
        markeredgewidth=0.6,
    )

axes[0].set_ylabel(
    "Centered additive contribution\n(log-odds scale)"
)

handles = [
    Line2D(
        [0],
        [0],
        color=MAIN_TARGET_COLORS[target],
        lw=2.0,
        label=TARGET_SHORT_NAMES[target],
    )
    for target in TARGET_COLUMNS
]
figure.legend(
    handles=handles,
    loc="upper center",
    ncol=2,
    frameon=False,
    bbox_to_anchor=(0.5, 1.005),
)
figure.tight_layout(
    rect=[0, 0, 1, 0.94]
)

figure2_meta = save_publication_figure(
    figure,
    "figure2_ebm_shape_functions_combined",
)
paper_ready_manifest_rows.append(figure2_meta)
print(figure2_meta)


In [ ]:
# Figure 3 — subgroup false-negative-rate gaps
row_specifications = [
    ("sex", "Male", "Male"),
    (
        "race_ethnicity",
        "Mexican American",
        "Mexican American",
    ),
    (
        "race_ethnicity",
        "Other Hispanic",
        "Other Hispanic",
    ),
    (
        "race_ethnicity",
        "Non-Hispanic Black",
        "Non-Hispanic Black",
    ),
    (
        "race_ethnicity",
        "Non-Hispanic Asian",
        "Non-Hispanic Asian",
    ),
    (
        "race_ethnicity",
        "Other or multiracial",
        "Other / multiracial",
    ),
    (
        "income_poverty_group",
        "Below 1",
        "IPR < 1",
    ),
    (
        "income_poverty_group",
        "1 to below 2",
        "IPR 1 to < 2",
    ),
    (
        "income_poverty_group",
        "2 to below 4",
        "IPR 2 to < 4",
    ),
    (
        "insurance_history",
        "Currently insured, past-year gap",
        "Past-year coverage gap",
    ),
    (
        "insurance_history",
        "Currently uninsured",
        "Currently uninsured",
    ),
]

row_order = [
    (variable, level)
    for variable, level, _ in row_specifications
]
row_labels = [
    label
    for _, _, label in row_specifications
]

plot_data = primary_fnr_gap_intervals.copy()
plot_data["row_key"] = list(
    zip(
        plot_data["subgroup_variable"],
        plot_data["subgroup"],
    )
)
plot_data["row_position"] = plot_data["row_key"].map(
    {
        key: position
        for position, key in enumerate(
            row_order[::-1]
        )
    }
)

all_lower = (
    100
    * plot_data["gap_ci_lower"].to_numpy(float)
)
all_upper = (
    100
    * plot_data["gap_ci_upper"].to_numpy(float)
)
x_min = 10 * np.floor(
    np.nanmin(all_lower) / 10
)
x_max = 10 * np.ceil(
    np.nanmax(all_upper) / 10
)

figure, axes = plt.subplots(
    1,
    2,
    figsize=(12.7, 7.6),
    sharey=True,
)

for panel_label, axis, target in zip(
    ["A", "B"],
    axes,
    TARGET_COLUMNS,
):
    target_data = plot_data.loc[
        plot_data["target"] == target
    ].copy()

    for model_name in MODEL_NAMES:
        subset = target_data.loc[
            target_data["model"] == model_name
        ].copy()
        subset["plot_y"] = (
            subset["row_position"]
            + (
                -0.12
                if model_name == "logistic"
                else 0.12
            )
        )

        estimate = (
            100
            * subset["gap_estimate"].to_numpy(float)
        )
        lower = (
            100
            * subset["gap_ci_lower"].to_numpy(float)
        )
        upper = (
            100
            * subset["gap_ci_upper"].to_numpy(float)
        )

        axis.errorbar(
            estimate,
            subset["plot_y"],
            xerr=np.vstack(
                [
                    estimate - lower,
                    upper - estimate,
                ]
            ),
            fmt=MODEL_MARKERS[model_name],
            markersize=5.4,
            capsize=2.8,
            color=MODEL_COLORS[model_name],
            ecolor=MODEL_COLORS[model_name],
            linewidth=1.2,
            label=MODEL_SHORT_NAMES[model_name],
            zorder=3,
        )

    axis.axvline(
        0,
        linestyle="--",
        linewidth=1,
        color="#666666",
    )
    axis.set_xlim(
        x_min - 2,
        x_max + 2,
    )
    axis.set_title(
        f"{panel_label}  {TARGET_SHORT_NAMES[target]}",
        loc="left",
        fontweight="semibold",
        pad=9,
    )
    axis.set_xlabel(
        "FNR gap (percentage points)"
    )
    axis.set_yticks(
        np.arange(len(row_order)),
        labels=row_labels[::-1],
    )
    clean_axis(
        axis,
        grid_axis="x",
    )

# Separators: insurance / income / race / sex.
for boundary in [1.5, 4.5, 9.5]:
    for axis in axes:
        axis.axhline(
            boundary,
            color="#C9C9C9",
            linewidth=0.75,
        )

handles = [
    Line2D(
        [0],
        [0],
        color=MODEL_COLORS[model_name],
        marker=MODEL_MARKERS[model_name],
        lw=1.2,
        label=MODEL_SHORT_NAMES[model_name],
    )
    for model_name in MODEL_NAMES
]
figure.legend(
    handles=handles,
    loc="upper center",
    ncol=2,
    frameon=False,
    bbox_to_anchor=(0.5, 1.005),
)
figure.tight_layout(
    rect=[0, 0, 1, 0.95]
)

figure3_meta = save_publication_figure(
    figure,
    "figure3_subgroup_fnr_gap_forest_plot",
)
paper_ready_manifest_rows.append(figure3_meta)
print(figure3_meta)


# Appendix Figure A2 — heatmap overview
heatmap_matrix = np.full(
    (
        len(row_order),
        len(MODEL_NAMES) * len(TARGET_COLUMNS),
    ),
    np.nan,
)
column_sequence = [
    ("logistic", TARGET_COLUMNS[0]),
    ("logistic", TARGET_COLUMNS[1]),
    ("ebm", TARGET_COLUMNS[0]),
    ("ebm", TARGET_COLUMNS[1]),
]
column_labels = [
    (
        f"{MODEL_SHORT_NAMES[model_name]}\n"
        f"{TARGET_SHORT_NAMES[target]}"
    )
    for model_name, target in column_sequence
]

for row_index, (
    subgroup_variable,
    level,
) in enumerate(row_order):
    for column_index, (
        model_name,
        target,
    ) in enumerate(column_sequence):
        estimate = plot_data.loc[
            (
                plot_data["subgroup_variable"]
                == subgroup_variable
            )
            & (plot_data["subgroup"] == level)
            & (plot_data["model"] == model_name)
            & (plot_data["target"] == target),
            "gap_estimate",
        ].iloc[0]
        heatmap_matrix[
            row_index,
            column_index,
        ] = 100 * estimate

max_abs = np.nanmax(
    np.abs(heatmap_matrix)
)
max_abs = (
    1.0
    if (
        not np.isfinite(max_abs)
        or max_abs == 0
    )
    else max_abs
)

heatmap_figure, heatmap_axis = plt.subplots(
    figsize=(8.7, 6.7)
)
image = heatmap_axis.imshow(
    heatmap_matrix,
    aspect="auto",
    vmin=-max_abs,
    vmax=max_abs,
    cmap="coolwarm",
)

heatmap_axis.set_xticks(
    np.arange(len(column_labels)),
    labels=column_labels,
)
heatmap_axis.set_yticks(
    np.arange(len(row_labels)),
    labels=row_labels,
)

for row_index in range(
    heatmap_matrix.shape[0]
):
    for column_index in range(
        heatmap_matrix.shape[1]
    ):
        heatmap_axis.text(
            column_index,
            row_index,
            (
                f"{heatmap_matrix[row_index, column_index]:+.1f}"
            ),
            ha="center",
            va="center",
            fontsize=8.5,
        )

colorbar = heatmap_figure.colorbar(
    image,
    ax=heatmap_axis,
    shrink=0.88,
)
colorbar.set_label(
    "FNR gap (percentage points)"
)
heatmap_figure.tight_layout()

appendix_heatmap_meta = save_publication_figure(
    heatmap_figure,
    "appendix_figureA2_subgroup_fnr_gap_heatmap",
)
paper_ready_manifest_rows.append(
    appendix_heatmap_meta
)
print(appendix_heatmap_meta)


In [ ]:
# Appendix Figure A3 — out-of-fold calibration curves

def calibration_points(
    outcome,
    probability,
    n_bins=10,
):
    calibration_data = pd.DataFrame(
        {
            "outcome": np.asarray(
                outcome,
                dtype=int,
            ),
            "probability": np.asarray(
                probability,
                dtype=float,
            ),
        }
    )

    calibration_data["bin"] = pd.qcut(
        calibration_data["probability"],
        q=n_bins,
        duplicates="drop",
    )

    return (
        calibration_data
        .groupby(
            "bin",
            observed=True,
        )
        .agg(
            mean_predicted=(
                "probability",
                "mean",
            ),
            observed_fraction=(
                "outcome",
                "mean",
            ),
            bin_n=(
                "outcome",
                "size",
            ),
        )
        .reset_index(drop=True)
    )


calibration_curves = {}
maximum_calibration_value = 0.0

for model_name in MODEL_NAMES:
    for target in TARGET_COLUMNS:
        curve = calibration_points(
            primary_probabilities[target],
            primary_probabilities[
                f"{model_name}__{target}__probability"
            ],
            n_bins=10,
        )
        calibration_curves[
            (model_name, target)
        ] = curve

        maximum_calibration_value = max(
            maximum_calibration_value,
            float(
                curve[
                    [
                        "mean_predicted",
                        "observed_fraction",
                    ]
                ].to_numpy().max()
            ),
        )

axis_limit = min(
    1.0,
    max(
        0.4,
        np.ceil(
            maximum_calibration_value * 10
        ) / 10,
    ),
)

figure, axes = plt.subplots(
    1,
    2,
    figsize=(10.8, 4.8),
    sharex=True,
    sharey=True,
)

for panel_label, axis, target in zip(
    ["A", "B"],
    axes,
    TARGET_COLUMNS,
):
    axis.plot(
        [0, axis_limit],
        [0, axis_limit],
        linestyle="--",
        linewidth=1,
        color="#777777",
        label="Ideal",
    )

    for model_name in MODEL_NAMES:
        curve = calibration_curves[
            (model_name, target)
        ]
        axis.plot(
            curve["mean_predicted"],
            curve["observed_fraction"],
            marker=MODEL_MARKERS[model_name],
            markersize=5,
            linewidth=1.5,
            color=MODEL_COLORS[model_name],
            label=MODEL_SHORT_NAMES[model_name],
        )

    axis.set_xlim(
        0,
        axis_limit,
    )
    axis.set_ylim(
        0,
        axis_limit,
    )
    axis.set_aspect(
        "equal",
        adjustable="box",
    )
    axis.set_title(
        f"{panel_label}  {TARGET_SHORT_NAMES[target]}",
        loc="left",
        fontweight="semibold",
        pad=9,
    )
    axis.set_xlabel(
        "Mean predicted probability"
    )
    clean_axis(
        axis,
        grid_axis="both",
    )

axes[0].set_ylabel(
    "Observed outcome proportion"
)

legend_handles = [
    Line2D(
        [0],
        [0],
        color="#777777",
        linestyle="--",
        lw=1,
        label="Ideal",
    ),
    *[
        Line2D(
            [0],
            [0],
            color=MODEL_COLORS[model_name],
            marker=MODEL_MARKERS[model_name],
            lw=1.5,
            label=MODEL_SHORT_NAMES[model_name],
        )
        for model_name in MODEL_NAMES
    ],
]
figure.legend(
    handles=legend_handles,
    loc="upper center",
    ncol=3,
    frameon=False,
    bbox_to_anchor=(0.5, 1.005),
)
figure.tight_layout(
    rect=[0, 0, 1, 0.94]
)

appendix_calibration_meta = save_publication_figure(
    figure,
    "appendix_figureA3_calibration_curves",
)
paper_ready_manifest_rows.append(
    appendix_calibration_meta
)
print(appendix_calibration_meta)


In [ ]:
# Appendix Figure A4 — sensitivity to omitting race/ethnicity

roc_change = race_omission_performance_with_intervals.loc[
    (
        race_omission_performance_with_intervals["metric"]
        == "roc_auc"
    )
].copy()

roc_order = [
    ("logistic", TARGET_COLUMNS[0]),
    ("logistic", TARGET_COLUMNS[1]),
    ("ebm", TARGET_COLUMNS[0]),
    ("ebm", TARGET_COLUMNS[1]),
]
roc_labels = [
    (
        f"{MODEL_SHORT_NAMES[model_name]} - "
        f"{TARGET_SHORT_NAMES[target]}"
    )
    for model_name, target in roc_order
]

race_gap_change = race_omission_fairness_with_intervals.loc[
    (
        race_omission_fairness_with_intervals[
            "metric"
        ]
        == "false_negative_rate"
    )
    & np.isclose(
        race_omission_fairness_with_intervals[
            "target_training_sensitivity"
        ],
        PRIMARY_TARGET_SENSITIVITY,
    )
    & (
        race_omission_fairness_with_intervals[
            "subgroup_variable"
        ]
        == "race_ethnicity"
    )
    & (
        race_omission_fairness_with_intervals[
            "subgroup"
        ]
        != race_omission_fairness_with_intervals[
            "reference_group"
        ]
    )
].copy()

race_levels = [
    level
    for level in RACE_ETHNICITY_LEVELS
    if level
    != REFERENCE_CATEGORIES["race_ethnicity"]
]
race_short_labels = {
    "Mexican American": "Mexican American",
    "Other Hispanic": "Other Hispanic",
    "Non-Hispanic Black": "Non-Hispanic Black",
    "Non-Hispanic Asian": "Non-Hispanic Asian",
    "Other or multiracial": "Other / multiracial",
}

figure, axes = plt.subplots(
    1,
    2,
    figsize=(13.2, 5.8),
    gridspec_kw={
        "width_ratios": [0.82, 1.35],
    },
)

# Panel A: change in ROC-AUC.
axis = axes[0]
roc_positions = np.arange(len(roc_order))[::-1]

for position, (
    model_name,
    target,
) in zip(
    roc_positions,
    roc_order,
):
    row = roc_change.loc[
        (roc_change["model"] == model_name)
        & (roc_change["target"] == target)
    ].iloc[0]

    estimate = (
        100
        * float(
            row["without_minus_with_estimate"]
        )
    )
    lower = (
        100
        * float(
            row["without_minus_with_ci_lower"]
        )
    )
    upper = (
        100
        * float(
            row["without_minus_with_ci_upper"]
        )
    )

    axis.errorbar(
        estimate,
        position,
        xerr=np.array(
            [
                [
                    estimate - lower,
                ],
                [
                    upper - estimate,
                ],
            ]
        ),
        fmt=MODEL_MARKERS[model_name],
        markersize=5.5,
        capsize=3,
        color=MAIN_TARGET_COLORS[target],
        ecolor=MAIN_TARGET_COLORS[target],
        linewidth=1.2,
        zorder=3,
    )

axis.axvline(
    0,
    linestyle="--",
    linewidth=1,
    color="#666666",
)
axis.set_yticks(
    roc_positions,
    labels=roc_labels,
)
axis.set_xlabel(
    "ROC-AUC change (percentage points)"
)
axis.set_title(
    "A  Discrimination",
    loc="left",
    fontweight="semibold",
    pad=9,
)
clean_axis(
    axis,
    grid_axis="x",
)

# Panel B: change in race-subgroup FNR gaps.
axis = axes[1]
race_positions = np.arange(
    len(race_levels)
)[::-1]

combination_offsets = {
    ("logistic", TARGET_COLUMNS[0]): -0.18,
    ("logistic", TARGET_COLUMNS[1]): -0.06,
    ("ebm", TARGET_COLUMNS[0]): 0.06,
    ("ebm", TARGET_COLUMNS[1]): 0.18,
}

for model_name, target in roc_order:
    subset = race_gap_change.loc[
        (race_gap_change["model"] == model_name)
        & (race_gap_change["target"] == target)
    ].set_index("subgroup")

    estimates = np.array(
        [
            100
            * float(
                subset.loc[
                    level,
                    "without_minus_with_gap_estimate",
                ]
            )
            for level in race_levels
        ]
    )
    lower = np.array(
        [
            100
            * float(
                subset.loc[
                    level,
                    "gap_difference_ci_lower",
                ]
            )
            for level in race_levels
        ]
    )
    upper = np.array(
        [
            100
            * float(
                subset.loc[
                    level,
                    "gap_difference_ci_upper",
                ]
            )
            for level in race_levels
        ]
    )

    axis.errorbar(
        estimates,
        race_positions
        + combination_offsets[
            (model_name, target)
        ],
        xerr=np.vstack(
            [
                estimates - lower,
                upper - estimates,
            ]
        ),
        fmt=MODEL_MARKERS[model_name],
        markersize=4.8,
        capsize=2.5,
        color=MAIN_TARGET_COLORS[target],
        ecolor=MAIN_TARGET_COLORS[target],
        linewidth=1.05,
        zorder=3,
    )

axis.axvline(
    0,
    linestyle="--",
    linewidth=1,
    color="#666666",
)
axis.set_yticks(
    race_positions,
    labels=[
        race_short_labels[level]
        for level in race_levels
    ],
)
axis.set_xlabel(
    "Change in FNR gap (percentage points)"
)
axis.set_title(
    "B  Race-subgroup error gaps",
    loc="left",
    fontweight="semibold",
    pad=9,
)
clean_axis(
    axis,
    grid_axis="x",
)

legend_handles = [
    Line2D(
        [0],
        [0],
        color=MAIN_TARGET_COLORS[target],
        marker=MODEL_MARKERS[model_name],
        lw=1.2,
        label=(
            f"{MODEL_SHORT_NAMES[model_name]} - "
            f"{TARGET_SHORT_NAMES[target]}"
        ),
    )
    for model_name, target in roc_order
]
figure.legend(
    handles=legend_handles,
    loc="upper center",
    ncol=4,
    frameon=False,
    bbox_to_anchor=(0.5, 1.015),
)
figure.tight_layout(
    rect=[0, 0, 1, 0.93]
)

appendix_race_omission_meta = save_publication_figure(
    figure,
    "appendix_figureA4_race_omission_sensitivity",
)
paper_ready_manifest_rows.append(
    appendix_race_omission_meta
)
print(appendix_race_omission_meta)


In [ ]:
# Add Appendix Figure A1 to the paper-ready output package
appendix_a1_stem = "appendix_figureA1_analytic_sample_flow"
appendix_a1_png = FIGURE_DIR / f"{appendix_a1_stem}.png"
appendix_a1_pdf = FIGURE_DIR / f"{appendix_a1_stem}.pdf"

for path in [appendix_a1_png, appendix_a1_pdf]:
    if not path.exists():
        raise FileNotFoundError(
            f"Appendix Figure A1 was not created: {path}"
        )
    destination = PAPER_READY_DIR / path.name
    destination.write_bytes(path.read_bytes())

# Keep the cell safe to rerun without duplicating the manifest row.
paper_ready_manifest_rows[:] = [
    row
    for row in paper_ready_manifest_rows
    if row.get("stem") != appendix_a1_stem
]

paper_ready_manifest_rows.append(
    {
        "artifact_type": "figure",
        "stem": appendix_a1_stem,
        "png": str(appendix_a1_png),
        "pdf": str(appendix_a1_pdf),
    }
)

print(
    "Added Appendix Figure A1 to paper-ready outputs:",
    appendix_a1_stem,
)


In [ ]:
paper_ready_outputs = save_publication_manifest(paper_ready_manifest_rows)
paper_ready_outputs


## Main-paper and appendix output manifest

In [ ]:
def copy_output(
    source_relative_path,
    destination_directory,
    role,
    description,
    required,
    destination_name=None,
):
    source = resolve_project_file(
        source_relative_path,
        required=required,
    )
    if source is None:
        return {
            "role": role,
            "description": description,
            "source": source_relative_path,
            "destination": "",
            "copied": False,
            "required": required,
            "sha256": "",
        }

    destination = destination_directory / (
        destination_name if destination_name else source.name
    )
    shutil.copy2(source, destination)
    return {
        "role": role,
        "description": description,
        "source": str(source),
        "destination": str(destination),
        "copied": True,
        "required": required,
        "sha256": sha256_file(destination),
    }


manifest_rows = []

main_specs = [
    (
        "outputs/tables/paper_table1_analytic_sample.csv",
        "Main table",
        "Table 1: analytic sample and target prevalence",
    ),
    (
        "outputs/tables/paper_table2_target_concordance.csv",
        "Main table",
        "Table 2: numerical target concordance",
    ),
    (
        "outputs/tables/paper_table3_primary_performance.csv",
        "Main table",
        "Table 3: primary predictive performance",
    ),
    (
        "outputs/figures/figure1_logistic_probability_effects.png",
        "Main figure",
        "Figure 1: logistic probability-scale contrasts",
    ),
    (
        "outputs/figures/figure1_logistic_probability_effects.pdf",
        "Main figure",
        "Figure 1: logistic probability-scale contrasts",
    ),
    (
        "outputs/figures/figure2_ebm_shape_functions_combined.png",
        "Main figure",
        "Figure 2: combined EBM shape functions",
    ),
    (
        "outputs/figures/figure2_ebm_shape_functions_combined.pdf",
        "Main figure",
        "Figure 2: combined EBM shape functions",
    ),
    (
        "outputs/figures/figure3_subgroup_fnr_gap_forest_plot.png",
        "Main figure",
        "Figure 3: subgroup FNR gap forest plot",
    ),
    (
        "outputs/figures/figure3_subgroup_fnr_gap_forest_plot.pdf",
        "Main figure",
        "Figure 3: subgroup FNR gap forest plot",
    ),
]

for source_path, role, description in main_specs:
    manifest_rows.append(
        copy_output(
            source_path,
            FINAL_MAIN_DIR,
            role,
            description,
            required=True,
        )
    )

required_appendix_specs = [
    (
        "outputs/tables/appendix_tableA1_primary_fnr_gaps_with_ci.csv",
        "Appendix table",
        "Appendix Table A1: subgroup FNR gaps with confidence intervals",
    ),
    (
        "outputs/tables/appendix_tableA2_threshold_sensitivity_fnr_gaps.csv",
        "Appendix table",
        "Appendix Table A2: threshold-sensitive subgroup FNR gaps",
    ),
    (
        "outputs/tables/appendix_tableA3_subgroup_positive_counts.csv",
        "Appendix table",
        "Appendix Table A3: target-positive subgroup counts",
    ),
    (
        "outputs/figures/appendix_figureA1_analytic_sample_flow.png",
        "Appendix figure",
        "Appendix Figure A1: analytic sample flow",
    ),
    (
        "outputs/figures/appendix_figureA1_analytic_sample_flow.pdf",
        "Appendix figure",
        "Appendix Figure A1: analytic sample flow",
    ),
    (
        "outputs/figures/appendix_figureA2_subgroup_fnr_gap_heatmap.png",
        "Appendix figure",
        "Appendix Figure A2: subgroup FNR heatmap",
    ),
    (
        "outputs/figures/appendix_figureA2_subgroup_fnr_gap_heatmap.pdf",
        "Appendix figure",
        "Appendix Figure A2: subgroup FNR heatmap",
    ),

    (
        "outputs/figures/appendix_figureA3_calibration_curves.png",
        "Appendix figure",
        "Appendix Figure A3: out-of-fold calibration curves",
    ),
    (
        "outputs/figures/appendix_figureA3_calibration_curves.pdf",
        "Appendix figure",
        "Appendix Figure A3: out-of-fold calibration curves",
    ),
    (
        "outputs/figures/appendix_figureA4_race_omission_sensitivity.png",
        "Appendix figure",
        "Appendix Figure A4: sensitivity to omitting race/ethnicity",
    ),
    (
        "outputs/figures/appendix_figureA4_race_omission_sensitivity.pdf",
        "Appendix figure",
        "Appendix Figure A4: sensitivity to omitting race/ethnicity",
    ),
    (
        "outputs/tables/threshold_sensitivity_stability_summary.csv",
        "Appendix table",
        "Threshold sensitivity summary",
    ),
    (
        "outputs/tables/sensitivity_complete_case_selection_summary.csv",
        "Appendix table",
        "Complete-case sensitivity summary",
    ),
    (
        "outputs/tables/sensitivity_weighted_unweighted_descriptive_comparison.csv",
        "Appendix table",
        "Weighted versus unweighted descriptive comparison",
    ),
    (
        "outputs/tables/sensitivity_model_class_performance.csv",
        "Appendix table",
        "Model-class sensitivity summary",
    ),
    (
        "outputs/tables/race_omission_probability_performance_with_intervals.csv",
        "Appendix table",
        "With-race versus without-race probability performance",
    ),
    (
        "outputs/tables/race_omission_probability_effect_comparison.csv",
        "Appendix table",
        "With-race versus without-race probability-scale effects",
    ),
    (
        "outputs/tables/race_omission_subgroup_changes_with_intervals.csv",
        "Appendix table",
        "With-race versus without-race subgroup fairness changes",
    ),
    (
        "outputs/tables/race_omission_threshold_stability_summary.csv",
        "Appendix table",
        "Threshold stability of race omission",
    ),
]

for source_path, role, description in required_appendix_specs:
    manifest_rows.append(
        copy_output(
            source_path,
            FINAL_APPENDIX_DIR,
            role,
            description,
            required=True,
        )
    )

optional_appendix_specs = [
    (
        "outputs/tables/paper_table1_analytic_sample.tex",
        "Appendix support",
        "Table 1 LaTeX export",
    ),
    (
        "outputs/tables/paper_table2_target_concordance.tex",
        "Appendix support",
        "Table 2 LaTeX export",
    ),
    (
        "outputs/tables/paper_table3_primary_performance.tex",
        "Appendix support",
        "Table 3 LaTeX export",
    ),
    (
        "outputs/tables/appendix_tableA1_primary_fnr_gaps_with_ci.tex",
        "Appendix support",
        "Appendix Table A1 LaTeX export",
    ),
    (
        "outputs/tables/appendix_tableA2_threshold_sensitivity_fnr_gaps.tex",
        "Appendix support",
        "Appendix Table A2 LaTeX export",
    ),
    (
        "outputs/tables/appendix_tableA3_subgroup_positive_counts.tex",
        "Appendix support",
        "Appendix Table A3 LaTeX export",
    ),
]

for source_path, role, description in optional_appendix_specs:
    manifest_rows.append(
        copy_output(
            source_path,
            FINAL_APPENDIX_DIR,
            role,
            description,
            required=False,
        )
    )

final_output_manifest = pd.DataFrame(manifest_rows)
FINAL_MANIFEST_PATH = FINAL_DIR / "final_output_manifest.csv"
final_output_manifest.to_csv(FINAL_MANIFEST_PATH, index=False)
final_output_manifest


## Concise sensitivity summary for the paper

This table identifies the essential result from each robustness check and points to the full appendix output.

In [ ]:
threshold_dependent_n = int(
    threshold_stability["threshold_stability_label"]
    .str.startswith("Threshold-dependent")
    .sum()
)

threshold_dependent_fnr = threshold_stability.loc[
    (threshold_stability["metric"] == "false_negative_rate")
    & (
        threshold_stability["subgroup"]
        != threshold_stability["reference_group"]
    )
    & threshold_stability[
        "threshold_stability_label"
    ].str.startswith(
        "Threshold-dependent",
        na=False,
    )
].copy()

threshold_dependent_fnr_n = int(
    len(threshold_dependent_fnr)
)
maximum_fnr_gap_range_pp = float(
    100 * threshold_dependent_fnr["gap_range"].max()
)

maximum_gap_range_pp = float(
    100 * threshold_stability["gap_range"].max()
)
maximum_weighting_difference = float(
    weighted_unweighted_sensitivity[
        "weighted_minus_unweighted"
    ].abs().max()
)

auc_changes = (
    race_omission_performance_with_intervals.loc[
        race_omission_performance_with_intervals[
            "metric"
        ]
        == "roc_auc"
    ]
)

maximum_absolute_auc_change = float(
    auc_changes[
        "without_minus_with_estimate"
    ].abs().max()
)

primary_fairness_changes = (
    race_omission_fairness_with_intervals.loc[
        np.isclose(
            race_omission_fairness_with_intervals[
                "target_training_sensitivity"
            ],
            PRIMARY_TARGET_SENSITIVITY,
        )
    ]
)

maximum_absolute_fnr_gap_change_pp = float(
    100
    * primary_fairness_changes.loc[
        primary_fairness_changes[
            "metric"
        ]
        == "false_negative_rate",
        "without_minus_with_gap_estimate",
    ].abs().max()
)

paper_sensitivity_summary = pd.DataFrame(
    [
        {
            "sensitivity_analysis": (
                "Threshold sensitivity"
            ),
            "required": True,
            "main_paper_value": (
                f"{threshold_dependent_fnr_n} subgroup FNR-gap "
                "patterns met the descriptive threshold-dependent "
                "rule; maximum FNR-gap range "
                f"{maximum_fnr_gap_range_pp:.1f} percentage points."
            ),
            "interpretation_caution": (
                "The stability label is descriptive, "
                "not a formal test."
            ),
            "full_output": (
                "appendix_tableA2_threshold_sensitivity_fnr_gaps.csv"
            ),
        },
        {
            "sensitivity_analysis": (
                "Complete-case and sample construction"
            ),
            "required": True,
            "main_paper_value": (
                f"{analytic_n:,} of {adult_n:,} adults "
                f"({100 * analytic_n / adult_n:.1f}%) "
                "were retained."
            ),
            "interpretation_caution": (
                "Complete-case selection limits "
                "generalisability; descriptive differences "
                "do not identify selection bias."
            ),
            "full_output": (
                "sensitivity_complete_case_exclusion_reasons.csv"
            ),
        },
        {
            "sensitivity_analysis": (
                "Weighted versus unweighted descriptions"
            ),
            "required": True,
            "main_paper_value": (
                "Largest absolute weighted-minus-unweighted "
                "difference on its stored scale: "
                f"{maximum_weighting_difference:.3f}."
            ),
            "interpretation_caution": (
                "MEC-weighted complete-case point estimates "
                "are not fully design-corrected population "
                "estimates."
            ),
            "full_output": (
                "sensitivity_weighted_unweighted_"
                "descriptive_comparison.csv"
            ),
        },
        {
            "sensitivity_analysis": (
                "Logistic regression versus additive EBM"
            ),
            "required": True,
            "main_paper_value": (
                "Performance, fixed probability contrasts, "
                "and primary subgroup gaps were compared "
                "across model classes."
            ),
            "interpretation_caution": (
                "Agreement supports robustness to functional "
                "form; disagreement should be reported rather "
                "than averaged away."
            ),
            "full_output": (
                "sensitivity_model_class_performance.csv"
            ),
        },
        {
            "sensitivity_analysis": (
                "With versus without race/ethnicity"
            ),
            "required": True,
            "main_paper_value": (
                "Maximum absolute ROC-AUC change "
                f"{maximum_absolute_auc_change:.3f}; maximum "
                "absolute change in an 80%-operating-point "
                "FNR gap "
                f"{maximum_absolute_fnr_gap_change_pp:.1f} "
                "percentage points."
            ),
            "interpretation_caution": (
                "Race omission is not equivalent to fairness "
                "and can change different metrics in different "
                "directions."
            ),
            "full_output": (
                "race_omission_subgroup_changes_"
                "with_intervals.csv"
            ),
        },
    ]
)

paper_sensitivity_summary.to_csv(
    FINAL_DIR / "paper_sensitivity_summary.csv",
    index=False,
)

paper_sensitivity_summary


## Reproducibility metadata and Notebook 09 checkpoint

In [ ]:
required_notebook09_outputs = [
    TABLE_DIR / "paper_table1_analytic_sample.csv",
    TABLE_DIR / "paper_table1_analytic_sample.html",
    TABLE_DIR / "paper_table1_analytic_sample.tex",
    TABLE_DIR / "paper_table2_target_concordance.csv",
    TABLE_DIR / "paper_table2_target_concordance.html",
    TABLE_DIR / "paper_table2_target_concordance.tex",
    TABLE_DIR / "paper_table3_primary_performance.csv",
    TABLE_DIR / "paper_table3_primary_performance.html",
    TABLE_DIR / "paper_table3_primary_performance.tex",
    TABLE_DIR / "appendix_tableA1_primary_fnr_gaps_with_ci.csv",
    TABLE_DIR / "appendix_tableA1_primary_fnr_gaps_with_ci.html",
    TABLE_DIR / "appendix_tableA1_primary_fnr_gaps_with_ci.tex",
    TABLE_DIR / "appendix_tableA2_threshold_sensitivity_fnr_gaps.csv",
    TABLE_DIR / "appendix_tableA2_threshold_sensitivity_fnr_gaps.html",
    TABLE_DIR / "appendix_tableA2_threshold_sensitivity_fnr_gaps.tex",
    TABLE_DIR / "appendix_tableA3_subgroup_positive_counts.csv",
    TABLE_DIR / "appendix_tableA3_subgroup_positive_counts.html",
    TABLE_DIR / "appendix_tableA3_subgroup_positive_counts.tex",
    FIGURE_DIR / "figure1_logistic_probability_effects.png",
    FIGURE_DIR / "figure1_logistic_probability_effects.pdf",
    FIGURE_DIR / "figure2_ebm_shape_functions_combined.png",
    FIGURE_DIR / "figure2_ebm_shape_functions_combined.pdf",
    FIGURE_DIR / "figure3_subgroup_fnr_gap_forest_plot.png",
    FIGURE_DIR / "figure3_subgroup_fnr_gap_forest_plot.pdf",
    FIGURE_DIR / "appendix_figureA1_analytic_sample_flow.png",
    FIGURE_DIR / "appendix_figureA1_analytic_sample_flow.pdf",
    FIGURE_DIR / "appendix_figureA2_subgroup_fnr_gap_heatmap.png",
    FIGURE_DIR / "appendix_figureA2_subgroup_fnr_gap_heatmap.pdf",
    FIGURE_DIR / "appendix_figureA3_calibration_curves.png",
    FIGURE_DIR / "appendix_figureA3_calibration_curves.pdf",
    FIGURE_DIR / "appendix_figureA4_race_omission_sensitivity.png",
    FIGURE_DIR / "appendix_figureA4_race_omission_sensitivity.pdf",
    FINAL_DIR / "final_output_manifest.csv",
    FINAL_DIR / "paper_sensitivity_summary.csv",
]

missing_outputs = [
    path for path in required_notebook09_outputs if not path.exists()
]
if missing_outputs:
    raise FileNotFoundError(
        "Notebook 09 did not create all required final outputs:\n"
        + "\n".join(f"- {path}" for path in missing_outputs)
    )

required_manifest_failures = final_output_manifest.loc[
    final_output_manifest["required"]
    & ~final_output_manifest["copied"]
]
if len(required_manifest_failures):
    raise FileNotFoundError(
        "At least one required main-paper or appendix output was not copied."
    )

metadata = {
    "analytic_sample_n": int(len(data)),
    "adult_analysis_base_n": int(len(analysis_base)),
    "complete_case_retention_share": float(len(data) / len(analysis_base)),
    "random_state": RANDOM_STATE,
    "outer_folds": N_SPLITS,
    "inner_threshold_folds": INNER_THRESHOLD_SPLITS,
    "target_sensitivity_levels": TARGET_SENSITIVITY_LEVELS,
    "primary_target_sensitivity": PRIMARY_TARGET_SENSITIVITY,
    "bootstrap_replicates": N_BOOTSTRAP,
    "target_columns": TARGET_COLUMNS,
    "primary_predictors": PRIMARY_PREDICTORS,
    "final_main_directory": str(FINAL_MAIN_DIR),
    "final_appendix_directory": str(FINAL_APPENDIX_DIR),
    "notebook08_sensitivity_outputs_loaded": True,
    "appendix_tableA3_calculated_from_analytic_data": True,
    "upstream_file_hashes": {
        "labelled_data": sha256_file(LABELLED_DATA_PATH),
        "fold_assignments": sha256_file(FOLD_ASSIGNMENT_PATH),
        "logistic_oof": sha256_file(LOGISTIC_OOF_PATH),
        "ebm_oof": sha256_file(EBM_OOF_PATH),
        "thresholded_oof": sha256_file(THRESHOLDED_LONG_PATH),
        "threshold_stability": sha256_file(THRESHOLD_STABILITY_PATH),
        "race_omission_performance": sha256_file(
            RACE_OMISSION_PERFORMANCE_PATH
        ),
        "race_omission_fairness": sha256_file(
            RACE_OMISSION_FAIRNESS_PATH
        ),
    },
    "software_versions": {
        "python": sys.version.split()[0],
        "pandas": pd.__version__,
        "numpy": np.__version__,
        "matplotlib": matplotlib.__version__,
        "interpret": interpret.__version__,
    },
}

NOTEBOOK09_METADATA_PATH = (
    PROCESSED_DIR / "final_outputs_metadata.json"
)
with NOTEBOOK09_METADATA_PATH.open("w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=2)

notebook09_checkpoint = {
    "analytic_sample_n": len(data),
    "appendix_tableA3_rows": len(appendix_tableA3),
    "appendix_tableA3_generated_from_data": True,
    "required_final_outputs_copied": len(required_manifest_failures) == 0,
    "all_required_notebook09_outputs_saved": len(missing_outputs) == 0,
    "metadata_saved": NOTEBOOK09_METADATA_PATH.exists(),
}

print("Saved Notebook 09 metadata to:")
print(NOTEBOOK09_METADATA_PATH)
print()
print("Notebook 09 checkpoint:")
notebook09_checkpoint


## Completion criteria

- All manuscript-facing tables are generated from upstream analysis data.
- All manuscript-facing figures are exported as PDF and PNG.
- The final output manifests and Notebook 09 checkpoint list every required artifact.